# CV Match Studio - the whole project in one Colab notebook

Rank **hundreds of CVs** against one job. **You upload the CVs in the web page this notebook starts** - nothing to upload in code.

**Run the cells top to bottom** (Runtime > Run all), then open the link printed by the *Launch* cell (about 3 minutes).

| What the app does | |
|---|---|
| Reads **PDF, DOCX, DOC, RTF, ODT, TXT, HTML, scanned images and ZIP** files | up to 500 CVs per run |
| Explainable **match %** (skills 50 / text 25 / experience 15 / education 10) | adjustable in the sidebar |
| Missing skills, **ATS score**, advice, best-fit role | per CV |
| Live progress, filters, must-have skills, blind screening, CSV / ZIP export | |

## 1. Install libraries and system tools

In [ ]:
!pip -q install streamlit pypdf striprtf nltk scikit-learn pandas joblib kagglehub
!apt-get -qq install -y antiword tesseract-ocr poppler-utils > /dev/null    # .doc files, scanned CVs, PDF tools

# OPTIONAL: meaning-based similarity (~90 MB). Set True, then switch it on in the app sidebar.
USE_SEMANTIC_MODEL = False
if USE_SEMANTIC_MODEL:
    !pip -q install sentence-transformers

## 2. Create the project files
Each cell writes one file. Open any cell to read the code.

In [ ]:
import os
os.makedirs("zarshan-project", exist_ok=True)
os.chdir("zarshan-project")
for d in ["resume_ai", ".streamlit", "models", "dataset", "samples", "outputs"]:
    os.makedirs(d, exist_ok=True)
print("Working in:", os.getcwd())

### 2a. Reading any file format

In [ ]:
%%writefile resume_ai/__init__.py
"""CV Match Studio - rank hundreds of CVs against one job, with explainable scores."""
__version__ = "3.0"

In [ ]:
%%writefile resume_ai/config.py
"""All settings in one place."""
import os
from pathlib import Path

ROOT = Path(__file__).resolve().parent.parent
OUT = Path("outputs")                                   # every run is saved in outputs/<timestamp>/
MODELS_DIR = ROOT / "models"

MAX_DOCS = int(os.environ.get("CV_MAX_DOCS", 500))      # most CVs per run (raise it if you have the RAM)
MAX_FILE_MB = 50                                        # biggest single file
MAX_ZIP_TOTAL_MB = 800                                  # biggest total size when unpacking a ZIP
READ_WORKERS = 6                                        # files read in parallel

# default weights of the four parts of the match score (re-scaled when a part does not apply)
WEIGHTS = {"skills": 0.50, "text": 0.25, "experience": 0.15, "education": 0.10}
SHORTLIST_AT = 65                                       # match % labelled "Shortlist"
REVIEW_AT = 40                                          # match % labelled "Review"

# what the upload box accepts
UPLOAD_TYPES = ["pdf", "docx", "doc", "rtf", "odt", "txt", "md", "html", "htm",
                "png", "jpg", "jpeg", "tif", "tiff", "bmp", "zip"]
PALETTE = ["#4f46e5", "#e11d48", "#059669", "#d97706", "#7c3aed", "#0891b2", "#db2777", "#65a30d"]

In [ ]:
%%writefile resume_ai/readers.py
"""Turn ANY resume file into plain text.

Supported (detected from the file's first bytes, so a wrongly named file still works):
  PDF            pypdf, then `pdftotext`, then OCR if the PDF is a scan (needs tesseract)
  DOCX           read straight from the XML - also gets text boxes, tables, headers
  DOC (old Word) antiword / catdoc / LibreOffice if installed, else a best-effort fallback
  RTF, ODT, HTML, TXT, MD, LOG, CSV
  PNG / JPG / TIFF / BMP images of a CV (OCR, needs tesseract)
  ZIP            expanded into the files inside it (see expand_zip)
Nothing is ever written to your disk except short-lived temp files for converters.
"""
from __future__ import annotations

import html as _html
import io
import os
import re
import shutil
import subprocess
import tempfile
import zipfile
from dataclasses import dataclass
from html.parser import HTMLParser
from pathlib import PurePosixPath
from typing import List, Tuple

from .config import MAX_FILE_MB, MAX_ZIP_TOTAL_MB

MAX_CHARS = 200_000
OLE_MAGIC = b"\xd0\xcf\x11\xe0\xa1\xb1\x1a\xe1"
TEXT_EXT = {".txt", ".md", ".log", ".csv", ".text", ".json"}
IMAGE_EXT = {".png", ".jpg", ".jpeg", ".tif", ".tiff", ".bmp"}


class ReadError(Exception):
    """A file that cannot be read - the message is shown to the user."""


@dataclass
class Read:
    text: str
    kind: str
    warning: str = ""


# ------------------------------------------------------------------ detection
def sniff(name: str, data: bytes) -> str:
    ext = os.path.splitext(name.lower())[1]
    head = data[:2048]
    if data[:5] == b"%PDF-" or b"%PDF-" in head[:1024]:
        return "pdf"
    if data[:2] == b"PK":
        try:
            names = set(zipfile.ZipFile(io.BytesIO(data)).namelist())
        except zipfile.BadZipFile:
            raise ReadError("The file is damaged (not a valid ZIP/DOCX).")
        if "word/document.xml" in names:
            return "docx"
        if "content.xml" in names and "META-INF/manifest.xml" in names:
            return "odt"
        if any(n.startswith(("xl/", "ppt/")) for n in names):
            raise ReadError("Excel/PowerPoint files are not resumes. Save the CV as PDF or DOCX.")
        return "zip"
    if data[:8] == OLE_MAGIC:
        if ext in (".xls", ".ppt"):
            raise ReadError("Excel/PowerPoint files are not resumes. Save the CV as PDF or DOCX.")
        return "doc"
    if data[:5] == b"{\\rtf":
        return "rtf"
    low = head.lower().lstrip()
    if low.startswith((b"<!doctype html", b"<html", b"<?xml")) or ext in (".html", ".htm"):
        return "html"
    if data[:8] == b"\x89PNG\r\n\x1a\n" or data[:3] == b"\xff\xd8\xff" or data[:4] in (b"II*\x00", b"MM\x00*") or data[:2] == b"BM":
        return "image"
    if ext in IMAGE_EXT:
        return "image"
    if ext in TEXT_EXT or _mostly_text(head):
        return "text"
    raise ReadError(f"Unsupported file type '{ext or 'unknown'}'. Use PDF, DOCX, DOC, RTF, ODT, TXT, HTML or an image.")


def _mostly_text(b: bytes) -> bool:
    if not b:
        return False
    printable = sum(1 for x in b if x in (9, 10, 13) or 32 <= x < 127 or x >= 160)
    return printable / len(b) > 0.9 and b"\x00" not in b


def decode_bytes(b: bytes) -> str:
    if b[:2] in (b"\xff\xfe", b"\xfe\xff"):
        return b.decode("utf-16", errors="ignore")
    for enc in ("utf-8-sig", "cp1252"):
        try:
            return b.decode(enc)
        except UnicodeDecodeError:
            continue
    return b.decode("latin-1", errors="ignore")


def _tidy(text: str) -> str:
    text = text.replace("\x00", "").replace("\r\n", "\n").replace("\r", "\n").replace("\xa0", " ")
    text = re.sub(r"[ \t]+\n", "\n", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()[:MAX_CHARS]


# ------------------------------------------------------------------ each format
def _pdf(data: bytes, ocr: bool) -> Tuple[str, str]:
    text, warn = "", ""
    try:
        try:
            from pypdf import PdfReader
        except ImportError:
            from PyPDF2 import PdfReader
        reader = PdfReader(io.BytesIO(data))
        if reader.is_encrypted:
            try:
                unlocked = bool(reader.decrypt(""))          # 0 = wrong password
            except Exception:
                unlocked = False
            if not unlocked:
                raise ReadError("The PDF is password protected.")
        parts = []
        for page in reader.pages[:25]:
            try:
                parts.append(page.extract_text() or "")
            except Exception:
                continue
        text = "\n".join(parts)
    except ReadError:
        raise
    except Exception:
        text = ""
    if len(text.strip()) < 40 and shutil.which("pdftotext"):
        text = _run_tool(["pdftotext", "-layout", "-l", "25", "{in}", "-"], data, ".pdf") or text
    if len(text.strip()) < 40:
        if ocr and shutil.which("tesseract") and shutil.which("pdftoppm"):
            text = _ocr_pdf(data)
            warn = ("Scanned PDF - text read with OCR (check it looks right)." if len(text.strip()) >= 40
                    else "No text could be read, even with OCR.")
        else:
            warn = "No text found - probably a scanned/image PDF. Install tesseract (OCR) or send a text-based PDF."
    return text, warn


def _wxml_to_text(xml: str) -> str:
    xml = re.sub(r"<mc:Fallback>.*?</mc:Fallback>", "", xml, flags=re.S)      # Word stores text boxes twice
    out = []
    for m in re.finditer(r"<w:t(?:\s[^>]*)?>([^<]*)</w:t>|(</w:p>)|(<w:tab\s*/>)|(<w:br[^>]*/>)|(<w:cr\s*/>)|(</w:tc>)", xml):
        if m.group(1) is not None:
            out.append(_html.unescape(m.group(1)))
        elif m.group(2) or m.group(4) or m.group(5):
            out.append("\n")
        else:
            out.append("\t")
    return "".join(out)


def _docx(data: bytes) -> str:
    try:
        z = zipfile.ZipFile(io.BytesIO(data))
    except zipfile.BadZipFile:
        raise ReadError("The DOCX file is damaged.")
    names = z.namelist()
    ordered = sorted(n for n in names if re.match(r"word/header\d*\.xml$", n)) + ["word/document.xml"] + \
        sorted(n for n in names if re.match(r"word/footer\d*\.xml$", n))
    return "\n".join(_wxml_to_text(z.read(n).decode("utf-8", "ignore")) for n in ordered if n in names)


def _odt(data: bytes) -> str:
    z = zipfile.ZipFile(io.BytesIO(data))
    xml = z.read("content.xml").decode("utf-8", "ignore")
    xml = re.sub(r"</text:(?:p|h)>|<text:line-break\s*/>", "\n", xml)
    xml = re.sub(r"<text:tab\s*/>", "\t", xml)
    return _html.unescape(re.sub(r"<[^>]+>", "", xml))


class _Strip(HTMLParser):
    BLOCK = {"p", "div", "br", "li", "tr", "h1", "h2", "h3", "h4", "h5", "h6", "section", "ul", "ol", "table"}

    def __init__(self):
        super().__init__()
        self.out, self.skip = [], 0

    def handle_starttag(self, tag, attrs):
        if tag in ("script", "style"):
            self.skip += 1
        if tag in self.BLOCK:
            self.out.append("\n")

    def handle_endtag(self, tag):
        if tag in ("script", "style") and self.skip:
            self.skip -= 1
        if tag in self.BLOCK:
            self.out.append("\n")

    def handle_data(self, d):
        if not self.skip:
            self.out.append(d)


def _html_text(data: bytes) -> str:
    p = _Strip()
    p.feed(decode_bytes(data))
    return "".join(p.out)


def _rtf(data: bytes) -> str:
    raw = data.decode("latin-1", errors="ignore")
    try:
        from striprtf.striprtf import rtf_to_text
        return rtf_to_text(raw, errors="ignore")
    except Exception:
        pass
    raw = re.sub(r"\\'([0-9a-fA-F]{2})", lambda m: bytes.fromhex(m.group(1)).decode("cp1252", "ignore"), raw)
    raw = re.sub(r"\\u(-?\d+)\??", lambda m: chr(int(m.group(1)) % 65536), raw)
    raw = re.sub(r"\{\\\*[^{}]*\}", "", raw)                                       # ignorable destinations
    raw = re.sub(r"\{\\(?:fonttbl|colortbl|stylesheet|info|pict)[^{}]*(?:\{[^{}]*\}[^{}]*)*\}", "", raw)
    raw = re.sub(r"\\(?:par|line|row)\b ?|\\sect\b ?", "\n", raw)
    raw = re.sub(r"\\tab\b ?|\\cell\b ?", "\t", raw)
    raw = re.sub(r"\\[a-zA-Z]+-?\d* ?", "", raw)
    return re.sub(r"[{}\\]", "", raw)


def _run_tool(cmd: List[str], data: bytes, suffix: str, timeout: int = 60) -> str:
    """Run an external converter on a temp copy of the file; returns '' on any problem."""
    with tempfile.TemporaryDirectory() as d:
        path = os.path.join(d, "f" + suffix)
        with open(path, "wb") as fh:
            fh.write(data)
        try:
            r = subprocess.run([c.replace("{in}", path) for c in cmd], capture_output=True, timeout=timeout)
            return decode_bytes(r.stdout) if r.returncode == 0 else ""
        except Exception:
            return ""


def _doc(data: bytes) -> Tuple[str, str]:
    for tool in ("antiword", "catdoc"):
        if shutil.which(tool):
            t = _run_tool([tool, "{in}"], data, ".doc", 30)
            if len(t.strip()) > 20:
                return t, ""
    office = shutil.which("soffice") or shutil.which("libreoffice")
    if office:
        with tempfile.TemporaryDirectory() as d:
            src = os.path.join(d, "f.doc")
            with open(src, "wb") as fh:
                fh.write(data)
            try:
                subprocess.run([office, "--headless", "--convert-to", "txt:Text", "--outdir", d, src],
                               capture_output=True, timeout=90,
                               env={**os.environ, "HOME": d})
                out = os.path.join(d, "f.txt")
                if os.path.exists(out):
                    t = decode_bytes(open(out, "rb").read())
                    if len(t.strip()) > 20:
                        return t, ""
            except Exception:
                pass
    return _doc_fallback(data), "Old .doc file read approximately (install 'antiword' for exact text, or save the CV as DOCX/PDF)."


def _doc_fallback(data: bytes) -> str:
    """No converter installed: pull readable text runs out of the binary. Rough but usually enough for skills."""
    chunks = [m.group().decode("cp1252", "ignore") for m in re.finditer(rb"[\x20-\x7e\xa0-\xff\t\r\n]{8,}", data)]
    chunks += [m.group().decode("utf-16le", "ignore") for m in re.finditer(rb"(?:[\x20-\x7e]\x00){8,}", data)]
    lines = []
    for c in chunks:
        for ln in re.split(r"[\r\n\x07]+", c):
            ln = ln.strip()
            letters = sum(ch.isalpha() for ch in ln)
            if len(ln) >= 8 and letters / len(ln) > 0.6 and re.search(r"[a-zA-Z]{3}", ln) and not re.search(r"[^\x20-\x7e]{3}", ln):
                lines.append(ln)
    return "\n".join(dict.fromkeys(lines))


def _ocr_image_bytes(data: bytes, suffix: str) -> str:
    if not shutil.which("tesseract"):
        raise ReadError("This is an image: OCR (tesseract) is not installed. Send the CV as PDF or DOCX.")
    return _run_tool(["tesseract", "{in}", "stdout"], data, suffix, timeout=90)


def _ocr_pdf(data: bytes) -> str:
    with tempfile.TemporaryDirectory() as d:
        src = os.path.join(d, "f.pdf")
        with open(src, "wb") as fh:
            fh.write(data)
        try:
            subprocess.run(["pdftoppm", "-r", "150", "-l", "3", "-png", src, os.path.join(d, "p")],
                           capture_output=True, timeout=90)
            texts = []
            for img in sorted(f for f in os.listdir(d) if f.endswith(".png")):
                r = subprocess.run(["tesseract", os.path.join(d, img), "stdout"], capture_output=True, timeout=90)
                texts.append(decode_bytes(r.stdout))
            return "\n".join(texts)
        except Exception:
            return ""


# ------------------------------------------------------------------ public API
def read_file(name: str, data: bytes, ocr: bool = True) -> Read:
    """File name + bytes -> Read(text, kind, warning). Raises ReadError with a friendly message."""
    if not data:
        raise ReadError("The file is empty.")
    if len(data) > MAX_FILE_MB * 1024 * 1024:
        raise ReadError(f"File is larger than {MAX_FILE_MB} MB.")
    kind = sniff(name, data)
    warn = ""
    try:
        if kind == "pdf":
            text, warn = _pdf(data, ocr)
        elif kind == "docx":
            text = _docx(data)
        elif kind == "doc":
            text, warn = _doc(data)
        elif kind == "odt":
            text = _odt(data)
        elif kind == "rtf":
            text = _rtf(data)
        elif kind == "html":
            text = _html_text(data)
        elif kind == "image":
            if not ocr:
                raise ReadError("This is an image and OCR is switched off.")
            text = _ocr_image_bytes(data, os.path.splitext(name)[1] or ".png")
            warn = "Image read with OCR (check it looks right)."
        elif kind == "zip":
            raise ReadError("ZIP inside a ZIP is not unpacked.")
        else:
            text = decode_bytes(data)
    except ReadError:
        raise
    except Exception as e:
        raise ReadError(f"Could not read this {kind.upper()} file ({type(e).__name__}).")
    text = _tidy(text)
    if len(text) < 20:
        raise ReadError(warn or "No readable text found in this file.")
    return Read(text, kind, warn)


def expand_zip(data: bytes, limit: int) -> List[Tuple[str, bytes]]:
    """Files inside a ZIP (one level). Skips folders, hidden/system files and anything too big."""
    out, total = [], 0
    try:
        z = zipfile.ZipFile(io.BytesIO(data))
    except zipfile.BadZipFile:
        raise ReadError("The ZIP file is damaged.")
    for info in z.infolist():
        base = PurePosixPath(info.filename).name
        if info.is_dir() or not base or base.startswith(("._", ".")) or "__MACOSX" in info.filename:
            continue
        if info.file_size > MAX_FILE_MB * 1024 * 1024:
            continue
        total += info.file_size
        if total > MAX_ZIP_TOTAL_MB * 1024 * 1024:
            raise ReadError(f"The ZIP is larger than {MAX_ZIP_TOTAL_MB} MB when unpacked.")
        if len(out) >= limit:
            break
        out.append((base, z.read(info)))
    return out

### 2b. Analysis: skills, matching, ATS, advice

In [ ]:
%%writefile resume_ai/preprocessing.py
"""Text cleaning and preprocessing shared by training and prediction.

IMPORTANT: the same clean_text() must be used when training the model AND when
predicting, otherwise the features won't line up.

v2 change: if NLTK (or its data files) is unavailable - e.g. no internet on first
run - we fall back to scikit-learn's built-in stop-word list and skip lemmatization
instead of crashing. Normal runs (NLTK available) behave exactly as before.
"""
import re

from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

try:
    import nltk
    from nltk.corpus import stopwords
    from nltk.stem import WordNetLemmatizer
except ImportError:  # NLTK not installed
    nltk = None


def _ensure_nltk():
    """Download the small NLTK data files once (quietly) if missing."""
    if nltk is None:
        return
    for pkg, path in [
        ("stopwords", "corpora/stopwords"),
        ("wordnet", "corpora/wordnet"),
        ("omw-1.4", "corpora/omw-1.4"),
    ]:
        try:
            nltk.data.find(path)
        except LookupError:
            try:
                nltk.download(pkg, quiet=True)
            except Exception:
                pass


def _setup():
    """Return (stopword_set, lemmatize_function) with graceful fallbacks."""
    _ensure_nltk()
    try:
        stop = set(stopwords.words("english"))
    except Exception:
        stop = set(ENGLISH_STOP_WORDS)
    try:
        lem = WordNetLemmatizer()
        lem.lemmatize("tests")  # raises LookupError if wordnet is missing
        return stop, lem.lemmatize
    except Exception:
        return stop, (lambda w: w)


STOPWORDS, _lemmatize = _setup()


def clean_text(text):
    """Lowercase, strip noise, remove stopwords and lemmatize.

    Returns a single space-joined string ready for TF-IDF.
    """
    if not isinstance(text, str):
        text = str(text)

    text = text.lower()
    text = re.sub(r"http\S+|www\.\S+", " ", text)        # URLs
    text = re.sub(r"\S+@\S+", " ", text)                  # emails
    text = re.sub(r"[^a-z\s]", " ", text)                 # keep letters only
    text = re.sub(r"\s+", " ", text).strip()              # collapse spaces

    tokens = [
        _lemmatize(tok)
        for tok in text.split()
        if tok not in STOPWORDS and len(tok) > 1
    ]
    return " ".join(tokens)

In [ ]:
%%writefile resume_ai/skills.py
"""Curated skill dictionary for each job category in the Kaggle
'Updated Resume Dataset' (25 categories), plus helpers to find which skills a
resume has and which it is missing (skill-gap analysis).
"""
import re

# category -> list of key skills employers look for
SKILLS_BY_CATEGORY = {
    "Data Science": [
        "python", "r", "machine learning", "deep learning", "statistics",
        "pandas", "numpy", "scikit-learn", "tensorflow", "sql", "data analysis",
        "data visualization", "nlp", "tableau", "power bi", "excel",
    ],
    "Python Developer": [
        "python", "django", "flask", "rest api", "sql", "git", "postgresql",
        "docker", "unit testing", "oop", "linux", "html",
    ],
    "Java Developer": [
        "java", "spring", "spring boot", "hibernate", "sql", "rest api",
        "maven", "git", "junit", "microservices", "javascript", "html",
    ],
    "Web Designing": [
        "html", "css", "javascript", "bootstrap", "photoshop", "figma",
        "responsive design", "ui", "ux", "jquery", "wordpress", "adobe xd",
    ],
    "DotNet Developer": [
        "c#", ".net", "asp.net", "sql server", "mvc", "rest api", "entity framework",
        "azure", "javascript", "git", "linq",
    ],
    "DevOps Engineer": [
        "docker", "kubernetes", "jenkins", "aws", "azure", "terraform", "ansible",
        "linux", "ci cd", "git", "bash", "monitoring",
    ],
    "Network Security Engineer": [
        "network security", "firewall", "cisco", "vpn", "tcp ip", "linux",
        "penetration testing", "ids", "ips", "wireshark", "encryption",
    ],
    "Database": [
        "sql", "mysql", "oracle", "postgresql", "mongodb", "plsql",
        "database design", "indexing", "backup", "etl", "stored procedures",
    ],
    "Hadoop": [
        "hadoop", "hive", "pig", "spark", "mapreduce", "hdfs", "sqoop",
        "kafka", "java", "sql", "big data",
    ],
    "ETL Developer": [
        "etl", "informatica", "sql", "data warehouse", "ssis", "talend",
        "oracle", "data modeling", "python", "unix",
    ],
    "Blockchain": [
        "blockchain", "solidity", "ethereum", "smart contracts", "web3",
        "cryptography", "hyperledger", "bitcoin", "javascript", "node js",
    ],
    "Automation Testing": [
        "selenium", "java", "testng", "automation", "junit", "cucumber",
        "api testing", "jenkins", "git", "python", "test cases",
    ],
    "Testing": [
        "manual testing", "test cases", "selenium", "jira", "bug tracking",
        "regression testing", "sql", "agile", "test plan", "qa",
    ],
    "SAP Developer": [
        "sap", "abap", "sap hana", "sap fico", "sap mm", "sap sd",
        "data dictionary", "smartforms", "idoc", "sql",
    ],
    "Business Analyst": [
        "business analysis", "requirements gathering", "sql", "excel", "agile",
        "scrum", "uml", "jira", "stakeholder management", "documentation",
        "power bi", "data analysis",
    ],
    "Operations Manager": [
        "operations management", "team management", "project management",
        "budgeting", "excel", "supply chain", "process improvement",
        "leadership", "kpi", "scheduling",
    ],
    "PMO": [
        "project management", "pmp", "ms project", "agile", "scrum", "budgeting",
        "risk management", "reporting", "stakeholder management", "excel",
    ],
    "Sales": [
        "sales", "negotiation", "crm", "lead generation", "communication",
        "marketing", "customer relationship", "cold calling", "salesforce",
        "target",
    ],
    "HR": [
        "recruitment", "onboarding", "payroll", "employee relations",
        "performance management", "hr policies", "communication", "excel",
        "talent acquisition", "training",
    ],
    "Advocate": [
        "legal research", "litigation", "drafting", "contract law",
        "case management", "compliance", "negotiation", "legal documentation",
        "court", "client counseling",
    ],
    "Arts": [
        "creativity", "drawing", "painting", "design", "photoshop",
        "illustration", "communication", "adobe", "sketching", "visual arts",
    ],
    "Health and fitness": [
        "fitness training", "nutrition", "anatomy", "exercise", "personal training",
        "physiotherapy", "wellness", "first aid", "communication", "diet planning",
    ],
    "Civil Engineer": [
        "autocad", "civil engineering", "construction", "structural analysis",
        "staad pro", "project management", "surveying", "estimation",
        "concrete", "site management",
    ],
    "Mechanical Engineer": [
        "autocad", "solidworks", "catia", "mechanical design", "ansys",
        "manufacturing", "thermodynamics", "cad", "project management", "gd t",
    ],
    "Electrical Engineering": [
        "circuit design", "matlab", "plc", "power systems", "autocad",
        "electrical machines", "control systems", "scada", "electronics",
        "embedded systems",
    ],
}


def _has_skill(skill, text_lower):
    """Word-boundary match so short skills like 'r' or 'c#' don't false-match."""
    pattern = r"(?<![a-z0-9])" + re.escape(skill) + r"(?![a-z0-9])"
    return re.search(pattern, text_lower) is not None


def match_skills(resume_text, category):
    """Return (matched, missing) skill lists for the given category."""
    skills = SKILLS_BY_CATEGORY.get(category, [])
    text_lower = resume_text.lower()
    matched, missing = [], []
    for skill in skills:
        if _has_skill(skill, text_lower):
            matched.append(skill)
        else:
            missing.append(skill)
    return matched, missing


def all_categories():
    return sorted(SKILLS_BY_CATEGORY.keys())


# =====================================================================
# v2: GLOBAL skill vocabulary - used to read the skills a JOB DESCRIPTION
# asks for (instead of only the fixed per-category lists above).
# =====================================================================
EXTRA_SKILLS = [
    # programming / web
    "c++", "typescript", "react", "angular", "vue", "node js", "express",
    "graphql", "fastapi", "django", "flask", "php", "laravel", "ruby", "golang",
    "rust", "scala", "kotlin", "swift", "android", "ios", "flutter", "matlab",
    "bash", "shell scripting", "data structures", "algorithms", "api development",
    # data / ai
    "pytorch", "keras", "xgboost", "matplotlib", "seaborn", "scipy", "opencv",
    "computer vision", "generative ai", "llm", "langchain", "hugging face",
    "airflow", "dbt", "snowflake", "bigquery", "redshift", "looker", "a/b testing",
    "feature engineering", "time series", "predictive modeling", "mlops",
    # cloud / devops
    "aws", "gcp", "redis", "elasticsearch", "nosql", "firebase", "github",
    "gitlab", "devops", "cloud computing", "serverless", "nginx",
    # business / other
    "agile", "scrum", "jira", "figma", "seo", "digital marketing",
    "google analytics", "content writing", "accounting", "tally", "quickbooks",
    "financial analysis", "market research", "customer service", "recruitment",
    # soft skills
    "leadership", "teamwork", "problem solving", "time management",
    "critical thinking", "presentation", "mentoring",
]

SOFT_SKILLS = {
    "communication", "leadership", "teamwork", "problem solving",
    "time management", "critical thinking", "stakeholder management",
    "negotiation", "creativity", "customer relationship", "team management",
    "presentation", "mentoring",
}

# generic words that appear in the category lists but are too vague to treat
# as a "skill" when reading a job description
_NOISY = {
    "target", "court", "training", "reporting", "documentation", "design",
    "monitoring", "automation", "exercise", "backup", "indexing", "ids", "ips",
    "drawing", "painting",
}

# canonical skill -> other ways people write it
ALIASES = {
    "javascript": ["js", "es6", "ecmascript"],
    "node js": ["node.js", "nodejs", "node"],
    "react": ["reactjs", "react.js"],
    "scikit-learn": ["sklearn", "scikit learn"],
    "machine learning": ["ml"],
    "nlp": ["natural language processing"],
    "deep learning": ["neural networks", "neural network"],
    "ci cd": ["ci/cd", "cicd", "continuous integration", "continuous delivery"],
    "tcp ip": ["tcp/ip"],
    "rest api": ["rest", "restful", "restful api", "rest apis", "restful apis"],
    "power bi": ["powerbi"],
    "postgresql": ["postgres"],
    "kubernetes": ["k8s"],
    "aws": ["amazon web services"],
    "gcp": ["google cloud", "google cloud platform"],
    "azure": ["microsoft azure"],
    "excel": ["microsoft excel", "ms excel"],
    "gd t": ["gd&t"],
    "spring boot": ["springboot"],
    "asp.net": ["aspnet", "asp .net"],
    "sql server": ["mssql", "ms sql", "microsoft sql server"],
    "ms project": ["microsoft project"],
    "data visualization": ["data visualisation", "data viz"],
    "unit testing": ["unit tests"],
    "oop": ["object oriented", "object-oriented", "object oriented programming"],
    "html": ["html5"],
    "css": ["css3"],
    "r": ["r programming", "rstudio", "r language"],   # bare "r" is too ambiguous
    "c#": ["c sharp"],
    "mongodb": ["mongo db"],
    "mysql": ["my sql"],
    "git": ["version control"],
    "problem solving": ["problem-solving"],
    "teamwork": ["team player", "collaboration", "collaborative"],
    "communication": ["communication skills", "verbal communication"],
    "llm": ["large language models", "large language model"],
    "generative ai": ["genai", "gen ai"],
    "computer vision": ["image processing"],
    "ui": ["user interface"],
    "ux": ["user experience"],
    "a/b testing": ["ab testing", "a/b tests"],
    "project management": ["program management"],
    "team management": ["people management", "managing teams"],
    "etl": ["elt"],
}


def _build_vocab():
    vocab = set(EXTRA_SKILLS) | SOFT_SKILLS
    for lst in SKILLS_BY_CATEGORY.values():
        vocab.update(lst)
    vocab -= _NOISY
    return sorted(vocab)


GLOBAL_SKILLS = _build_vocab()


def _compile_patterns():
    pats = {}
    for canon in GLOBAL_SKILLS:
        forms = ALIASES["r"] if canon == "r" else [canon] + ALIASES.get(canon, [])
        forms = sorted({f for f in forms if len(f) > 1}, key=len, reverse=True)
        if forms:
            body = "|".join(re.escape(f) for f in forms)
            pats[canon] = re.compile(r"(?<![a-z0-9])(?:" + body + r")(?![a-z0-9])")
    return pats


_PATTERNS = _compile_patterns()


def extract_skills(text):
    """Return the sorted list of known skills (canonical names) found in text."""
    text_lower = (text or "").lower()
    return sorted(c for c, p in _PATTERNS.items() if p.search(text_lower))


def is_soft(skill):
    return skill in SOFT_SKILLS

In [ ]:
%%writefile resume_ai/extractor.py
"""Pull structured facts out of raw resume / job-description text:
contact info, sections, years of experience and education level.

Deliberately NOT extracted: name, gender, age, photo, address. The scoring never
sees them, which keeps the screening focused on skills and experience.
"""
import re
from datetime import date

# ---------------------------------------------------------------- contacts
_EMAIL = re.compile(r"[\w.+-]+@[\w-]+\.[\w.-]+")
_PHONE = re.compile(r"(?<!\d)(?:\+?\d[\d\s().-]{8,}\d)(?!\d)")
_LINKEDIN = re.compile(r"linkedin\.com/[^\s,;]+", re.I)
_GITHUB = re.compile(r"github\.com/[^\s,;]+", re.I)


def extract_contacts(text):
    phones = [p.strip() for p in _PHONE.findall(text)
              if 10 <= len(re.sub(r"\D", "", p)) <= 15]
    return {
        "email": bool(_EMAIL.search(text)),
        "phone": bool(phones),
        "linkedin": bool(_LINKEDIN.search(text)),
        "github": bool(_GITHUB.search(text)),
    }


# ---------------------------------------------------------------- sections
SECTION_KEYWORDS = {
    "summary": ["summary", "objective", "profile", "about me", "career objective",
                "professional summary"],
    "experience": ["experience", "work experience", "professional experience",
                   "employment history", "work history", "internship", "internships"],
    "education": ["education", "academic background", "qualifications",
                  "educational qualification", "academics"],
    "skills": ["skills", "technical skills", "key skills", "core competencies",
               "skill set", "technologies"],
    "projects": ["projects", "academic projects", "personal projects"],
    "certifications": ["certifications", "certificates", "courses"],
    "achievements": ["achievements", "awards", "accomplishments", "honors"],
}


def _heading(line):
    norm = re.sub(r"[^a-z ]", " ", line.lower())
    norm = re.sub(r"\s+", " ", norm).strip()
    if not norm or len(norm.split()) > 4:
        return None
    for name, keys in SECTION_KEYWORDS.items():
        for k in keys:
            if norm == k or norm.startswith(k + " ") or norm.endswith(" " + k):
                return name
    return None


def split_sections(text):
    """Return {section_name: text}. Lines before the first heading go to 'header'."""
    sections, current, buf = {}, "header", []
    for line in text.splitlines():
        h = _heading(line)
        if h:
            sections[current] = sections.get(current, "") + "\n".join(buf) + "\n"
            current, buf = h, []
        else:
            buf.append(line)
    sections[current] = sections.get(current, "") + "\n".join(buf)
    return sections


def detect_sections(text):
    """Set of section names present (by heading, or by the word appearing at all
    for single-block resumes that have no line breaks)."""
    found = {k for k in split_sections(text) if k != "header"}
    low = text.lower()
    for k in ("experience", "education", "skills", "projects"):
        if k not in found and re.search(r"\b" + k + r"\b", low):
            found.add(k)
    return found


# ---------------------------------------------------------------- experience
_MONTH = r"(?:jan|feb|mar|apr|may|jun|jul|aug|sep|oct|nov|dec)[a-z]*\.?\s*,?\s*"
_RANGE = re.compile(
    r"(?:" + _MONTH + r")?((?:19|20)\d{2})\s*(?:-|–|—|to)\s*(?:" + _MONTH +
    r")?((?:19|20)\d{2}|present|current|now|till date|ongoing)", re.I)
_EXPLICIT_A = re.compile(
    r"(\d{1,2}(?:\.\d)?)\s*\+?\s*(?:years?|yrs?)\s*(?:of\s+)?(?:[a-z]+\s+){0,4}?"
    r"(?:experience|exp)\b")
_EXPLICIT_B = re.compile(
    r"experience\s*(?:of|:|-)?\s*(?:over\s+|around\s+|about\s+)?(\d{1,2}(?:\.\d)?)\s*\+?\s*(?:years?|yrs?)")


def years_of_experience(text):
    """Best-effort total years of work experience (float)."""
    low = text.lower()
    explicit = [float(m) for m in _EXPLICIT_A.findall(low) + _EXPLICIT_B.findall(low)]
    explicit = [e for e in explicit if e <= 45]
    if explicit:
        return max(explicit)

    # otherwise add up date ranges - but only inside the experience section so
    # that education years (2016-2020) are not counted as work.
    secs = split_sections(text)
    scope = secs.get("experience") or "\n".join(
        l for k, v in secs.items() if k not in ("education", "certifications")
        for l in v.splitlines())
    this_year = date.today().year
    spans = []
    for a, b in _RANGE.findall(scope):
        start = int(a)
        end = this_year if not b.isdigit() else int(b)
        if 0 <= end - start <= 40 and end >= start:
            spans.append((start, end))
    if not spans:
        return 0.0
    spans.sort()
    merged = [list(spans[0])]
    for s, e in spans[1:]:
        if s <= merged[-1][1]:
            merged[-1][1] = max(merged[-1][1], e)
        else:
            merged.append([s, e])
    return float(sum(e - s for s, e in merged))


def required_years(jd_text):
    """Years of experience a job description asks for (None if not stated)."""
    low = jd_text.lower()
    for m in re.finditer(r"(\d{1,2})\s*\+?\s*(?:(?:to|-)\s*\d{1,2}\s*)?(?:years?|yrs?)", low):
        ctx = low[max(0, m.start() - 60): m.end() + 60]
        if "experience" in ctx or "exp" in ctx:
            return float(m.group(1))
    return None


# ---------------------------------------------------------------- education
EDU_LEVELS = [
    (4, "PhD", r"\b(?:ph\.?d|doctorate|doctoral)\b"),
    (3, "Master's", r"\b(?:master'?s?|m\.sc|msc|m\.tech|mtech|m\.s\b|ms in|mba|m\.e\b|mca|m\.com)"),
    (2, "Bachelor's", r"\b(?:bachelor'?s?|b\.sc|bsc|b\.tech|btech|b\.e\b|bca|bba|b\.com|b\.a\b|undergraduate|bs in)"),
    (1, "Diploma", r"\b(?:diploma|associate degree)\b"),
    (0, "High school", r"\b(?:high school|intermediate|hsc|ssc|matriculation)\b"),
]


def education_level(text, lowest=False):
    """(level_number, label) of the highest education found, or (None, None).

    lowest=True is used for job descriptions: "Bachelor's or Master's" means a
    Bachelor's is enough, so we take the lowest level mentioned.
    """
    low = text.lower()
    hits = [(lvl, label) for lvl, label, pat in EDU_LEVELS if re.search(pat, low)]
    if not hits:
        return None, None
    return min(hits) if lowest else max(hits)


def build_profile(text):
    """Everything we can say about a resume without a job description."""
    lvl, label = education_level(text)
    return {
        "years_experience": years_of_experience(text),
        "education_level": lvl,
        "education_label": label,
        "contacts": extract_contacts(text),
        "sections": sorted(detect_sections(text)),
        "word_count": len(text.split()),
    }

In [ ]:
%%writefile resume_ai/matcher.py
"""Resume <-> Job-description matching.

The final match % is a weighted blend of four understandable parts:

    Skills match        50 %   skills the JD asks for vs skills found in the resume
                               ("required" = 1.0, "preferred" = 0.5, soft skills count half)
    Text similarity     25 %   TF-IDF cosine (+ semantic embeddings if enabled)
    Experience          15 %   resume years vs years the JD asks for
    Education           10 %   resume degree level vs degree the JD asks for

If the JD does not mention a part (e.g. no years stated) that part is dropped and the
remaining weights are re-scaled. The job description is read ONCE (prepare_job) and then
reused for every CV, so ranking hundreds of CVs stays fast.
"""
from __future__ import annotations

import re
from collections import Counter
from dataclasses import dataclass
from typing import Dict, List, Optional

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

from .config import WEIGHTS
from .extractor import education_level, required_years, years_of_experience
from .preprocessing import clean_text
from .skills import extract_skills, is_soft

TFIDF_FULL_SCORE_AT = 0.45     # cosine that we treat as a "perfect" text match
_PREF_HINT = re.compile(r"preferred|nice to have|good to have|bonus|desirable|a plus|optional", re.I)
_REQ_HEAD = re.compile(r"requirements?|required|must have|qualifications|responsibilit|what you|you will|you have", re.I)
_JD_FILLER = {
    "role", "hiring", "requirement", "requirements", "year", "years", "strong",
    "preferred", "experience", "ability", "work", "skill", "skills", "team",
    "looking", "join", "candidate", "responsibility", "responsibilities",
    "familiarity", "excellent", "knowledge", "understanding", "nice", "have",
    "using", "company", "position", "job", "opportunity", "about", "must",
    "required", "plus", "bonus", "good", "great", "including", "within",
    "across", "highly", "proven", "demonstrated", "degree", "field", "ideal",
    "build", "building", "business", "environment", "will", "want", "bachelor",
    "bachelors", "master", "masters", "quantitative",
}


@dataclass
class JobSpec:
    """Everything we read from the job description, computed once."""
    text: str
    clean: str
    required: List[str]
    preferred: List[str]
    need_years: Optional[float]
    need_lvl: Optional[int]
    need_label: Optional[str]
    role: Optional[str] = None          # job family the JD reads like (filled in by the Job)


def verdict(score: float) -> str:
    if score >= 75:
        return "Excellent match"
    if score >= 55:
        return "Good match"
    if score >= 35:
        return "Partial match"
    return "Weak match"


def split_jd_skills(jd_text: str):
    """Return (required, preferred) skill lists found in a job description."""
    required, preferred = set(), set()
    mode = "required"
    for line in jd_text.splitlines():
        stripped = line.strip()
        if not stripped:
            continue
        short = len(stripped) < 80
        if short and _PREF_HINT.search(stripped):
            mode = "preferred"
        elif short and _REQ_HEAD.search(stripped):
            mode = "required"
        line_mode = "preferred" if _PREF_HINT.search(stripped) else mode
        for s in extract_skills(stripped):
            (required if line_mode == "required" else preferred).add(s)
    preferred -= required
    return sorted(required), sorted(preferred)


def prepare_job(jd_text: str) -> JobSpec:
    req, pref = split_jd_skills(jd_text)
    lvl, label = education_level(jd_text, lowest=True)
    return JobSpec(text=jd_text, clean=clean_text(jd_text), required=req, preferred=pref,
                   need_years=required_years(jd_text), need_lvl=lvl, need_label=label)


def _tfidf_cosine(a_clean: str, b_clean: str) -> float:
    if not a_clean or not b_clean:
        return 0.0
    try:
        m = TfidfVectorizer(ngram_range=(1, 2), sublinear_tf=True).fit_transform([a_clean, b_clean])
        return float(cosine_similarity(m[0], m[1])[0][0])
    except ValueError:
        return 0.0


def _missing_keywords(resume_clean: str, jd_clean: str, skill_words: set, top: int = 12) -> List[str]:
    resume_tokens = set(resume_clean.split())
    counts = Counter(t for t in jd_clean.split() if len(t) > 3)
    out = []
    for word, _ in counts.most_common(60):
        if word in resume_tokens or word in skill_words or word in _JD_FILLER:
            continue
        out.append(word)
        if len(out) >= top:
            break
    return out


def match(resume_text: str, job, weights: Optional[Dict[str, float]] = None,
          sem: Optional[float] = None, resume_clean: Optional[str] = None) -> dict:
    """Compare one resume with a job (JobSpec or raw text). `sem` = optional embedding cosine."""
    spec = job if isinstance(job, JobSpec) else prepare_job(job)
    W = {**WEIGHTS, **(weights or {})}
    required, preferred = spec.required, spec.preferred
    resume_skills = set(extract_skills(resume_text))

    # ---- 1. skills
    skill_weights = {s: (1.0 if s in required else 0.5) * (0.5 if is_soft(s) else 1.0)
                     for s in required + preferred}
    skills_score, matched_req, missing_req, matched_pref, missing_pref = None, [], [], [], []
    if skill_weights:
        got = sum(w for s, w in skill_weights.items() if s in resume_skills)
        skills_score = got / sum(skill_weights.values())
        matched_req = [s for s in required if s in resume_skills]
        missing_req = [s for s in required if s not in resume_skills]
        matched_pref = [s for s in preferred if s in resume_skills]
        missing_pref = [s for s in preferred if s not in resume_skills]

    # ---- 2. text similarity
    r_clean = resume_clean if resume_clean is not None else clean_text(resume_text)
    tfidf_cos = _tfidf_cosine(r_clean, spec.clean)
    text_score = min(1.0, tfidf_cos / TFIDF_FULL_SCORE_AT)
    if sem is not None:
        text_score = 0.5 * text_score + 0.5 * max(0.0, min(1.0, (sem - 0.15) / 0.55))

    # ---- 3. experience
    have_years = years_of_experience(resume_text)
    need_years = spec.need_years
    exp_score = None if need_years is None else (min(1.0, have_years / need_years) if need_years else 1.0)

    # ---- 4. education
    have_lvl, have_label = education_level(resume_text)
    edu_score = None
    if spec.need_lvl is not None:
        edu_score = 0.0 if have_lvl is None else (1.0 if have_lvl >= spec.need_lvl
                                                   else (have_lvl + 1) / (spec.need_lvl + 1))

    parts = {"skills": skills_score, "text": text_score, "experience": exp_score, "education": edu_score}
    active = {k: v for k, v in parts.items() if v is not None and W.get(k, 0) > 0}
    total_w = sum(W[k] for k in active) or 1.0
    score = 100 * sum(W[k] * v for k, v in active.items()) / total_w

    details = {
        "skills": (f"{len(matched_req) + len(matched_pref)} of {len(skill_weights)} skills from the JD found"
                   if skill_weights else "No recognisable skills in the job description"),
        "text": "Wording overlap" + (" + meaning similarity" if sem is not None else ""),
        "experience": ("JD does not state years" if need_years is None
                       else f"{have_years:g} yrs found vs {need_years:g} asked"),
        "education": ("JD does not state a degree" if spec.need_lvl is None
                      else f"{have_label or 'none found'} vs {spec.need_label} asked"),
    }
    names = {"skills": "Skills", "text": "Text similarity", "experience": "Experience", "education": "Education"}
    breakdown = [{
        "key": k, "name": names[k], "detail": details[k],
        "weight": round(100 * W[k] / total_w) if k in active else 0,
        "score": round(100 * parts[k]) if parts[k] is not None else None,
    } for k in names]

    skill_words = {w for s in required + preferred for w in s.split()}
    return {
        "score": round(score, 1), "verdict": verdict(score), "breakdown": breakdown,
        "required_skills": required, "preferred_skills": preferred,
        "matched_required": matched_req, "missing_required": missing_req,
        "matched_preferred": matched_pref, "missing_preferred": missing_pref,
        "extra_skills": sorted(resume_skills - set(required) - set(preferred))[:25],
        "missing_keywords": _missing_keywords(r_clean, spec.clean, skill_words),
        "years_have": have_years, "years_need": need_years,
        "semantic_used": sem is not None, "tfidf_cosine": round(tfidf_cos, 3),
    }

In [ ]:
%%writefile resume_ai/semantic.py
"""OPTIONAL meaning-based similarity using a small sentence-embedding model (MiniLM).

TF-IDF only sees identical words, so "built ML models" vs "machine learning engineer"
scores low. Embeddings understand they are close. Needs `pip install sentence-transformers`
(first use downloads ~90 MB). All CVs are embedded in ONE batch, so it stays quick.
"""
from __future__ import annotations

import importlib.util
from typing import List, Optional

import numpy as np

MODEL = "sentence-transformers/all-MiniLM-L6-v2"


def available() -> bool:
    return importlib.util.find_spec("sentence_transformers") is not None


class Embedder:
    def __init__(self, name: str = MODEL):
        self.model, self.error, self.name = None, "", name.split("/")[-1]
        try:
            from sentence_transformers import SentenceTransformer
            try:
                import torch
                dev = "cuda" if torch.cuda.is_available() else "cpu"
            except Exception:
                dev = "cpu"
            self.model = SentenceTransformer(name, device=dev)
            self.model.max_seq_length = 256
        except Exception as e:
            self.error = str(e)

    def encode_docs(self, texts: List[str], chunk_words: int = 150, max_chunks: int = 12) -> Optional[np.ndarray]:
        """One normalised vector per document (mean of its chunks). None if the model is unavailable."""
        if self.model is None:
            return None
        chunks, owner = [], []
        for i, t in enumerate(texts):
            words = t.split() or [""]
            for j in list(range(0, len(words), chunk_words))[:max_chunks]:
                chunks.append(" ".join(words[j:j + chunk_words]))
                owner.append(i)
        vecs = self.model.encode(chunks, batch_size=128, normalize_embeddings=True, show_progress_bar=False)
        owner = np.array(owner)
        out = np.vstack([vecs[owner == i].mean(0) for i in range(len(texts))])
        norms = np.linalg.norm(out, axis=1, keepdims=True)
        return (out / np.maximum(norms, 1e-9)).astype(np.float32)

In [ ]:
%%writefile resume_ai/ats.py
"""ATS-readability check: would an Applicant Tracking System (and a human skimming
for 10 seconds) be able to find the important things in this resume?

Score is out of 100. Every check carries a plain-language tip.
"""
import re

from .extractor import detect_sections, extract_contacts

ACTION_VERBS = {
    "led", "developed", "built", "designed", "implemented", "managed", "created",
    "improved", "increased", "reduced", "delivered", "launched", "optimized",
    "optimised", "automated", "analyzed", "analysed", "deployed", "achieved",
    "coordinated", "trained", "negotiated", "engineered", "migrated",
    "collaborated", "mentored", "streamlined", "architected", "resolved",
    "generated", "spearheaded", "established", "maintained", "executed",
    "integrated", "tested", "researched", "presented",
}
_NUMBER_WINS = re.compile(
    r"\d+(?:\.\d+)?\s*%|[$€£₹]\s?\d|\b\d+(?:\.\d+)?\s*[kKmMx]\b|"
    r"\b\d{2,}\+?\s+(?:users|clients|customers|projects|records|models|members|"
    r"people|applications|apps|systems|reports|tickets|leads|candidates)\b")
_BULLET = re.compile(r"^\s*[•●▪◦·*\-–>]\s+", re.M)


def _status(points, max_points):
    if points >= max_points:
        return "pass"
    return "partial" if points > 0 else "fail"


def ats_check(text):
    contacts = extract_contacts(text)
    sections = detect_sections(text)
    words = text.split()
    n_words = len(words)
    low = text.lower()
    checks = []

    def add(name, points, max_points, tip):
        checks.append({"name": name, "points": points, "max": max_points,
                       "status": _status(points, max_points), "tip": tip})

    add("Email address", 10 if contacts["email"] else 0, 10,
        "Add a professional email at the top - recruiters and ATS tools look for it first.")
    add("Phone number", 10 if contacts["phone"] else 0, 10,
        "Add a phone number with country code.")
    add("LinkedIn or GitHub link", 5 if (contacts["linkedin"] or contacts["github"]) else 0, 5,
        "Link your LinkedIn or GitHub (or portfolio) so employers can verify your work.")

    for key, label in (("experience", "Experience"), ("education", "Education"),
                       ("skills", "Skills")):
        add(f"{label} section", 10 if key in sections else 0, 10,
            f"Add a clearly titled '{label}' heading - ATS parsers split resumes by heading.")
    add("Projects or certifications",
        5 if ({"projects", "certifications"} & sections) else 0, 5,
        "Add a Projects or Certifications section - it is the best way to show skills "
        "when work experience is short.")

    if 250 <= n_words <= 1100:
        pts = 10
    elif 150 <= n_words < 250 or 1100 < n_words <= 1500:
        pts = 5
    else:
        pts = 0
    add(f"Length ({n_words} words)", pts, 10,
        "Aim for roughly 300-900 words (1-2 pages). Too short looks thin; too long gets skimmed.")

    wins = len(_NUMBER_WINS.findall(text))
    add(f"Measurable results ({wins} found)", 10 if wins >= 3 else 5 if wins >= 1 else 0, 10,
        "Add numbers to your achievements, e.g. 'cut report time by 40%' or "
        "'built a model used by 200+ customers'.")

    verbs = sum(1 for w in re.findall(r"[a-z]+", low) if w in ACTION_VERBS)
    add(f"Action verbs ({verbs} found)", 10 if verbs >= 6 else 5 if verbs >= 3 else 0, 10,
        "Start bullet points with strong verbs: built, led, automated, reduced, deployed...")

    bullets = len(_BULLET.findall(text))
    lines = len([l for l in text.splitlines() if l.strip()])
    add("Scannable structure", 5 if bullets >= 4 or lines >= 25 else 0, 5,
        "Use short bullet points instead of long paragraphs.")

    pronouns = len(re.findall(r"\b(?:i|my|me)\b", low))
    add("Professional tone", 5 if pronouns <= 4 else 0, 5,
        "Avoid 'I' and 'my' - write 'Built X' instead of 'I built X'.")

    score = sum(c["points"] for c in checks)
    return {"score": score, "checks": checks}

In [ ]:
%%writefile resume_ai/advice.py
"""Turn the raw numbers into plain-language next steps for the job seeker."""


def build_advice(profile, ats, match=None):
    tips = []
    if match:
        s = match["score"]
        if match["missing_required"]:
            top = ", ".join(match["missing_required"][:6])
            tips.append(
                f"The job asks for {top}, which your resume does not mention. If you really "
                "have this experience, add it to your Skills section and name a project or job "
                "where you used it. If you don't, treat it as your learning list - don't add "
                "skills you can't back up in an interview.")
        if match["years_need"] and match["years_have"] < match["years_need"]:
            tips.append(
                f"The role asks for about {match['years_need']:g} years of experience and we "
                f"found {match['years_have']:g}. Make internships, freelance work and serious "
                "projects visible, with dates, so they count.")
        if match["missing_keywords"]:
            tips.append("Words this job uses often but your resume doesn't: "
                        + ", ".join(match["missing_keywords"][:8])
                        + ". Use the ones that honestly describe your work.")
        if s >= 75:
            tips.append("Strong match - apply. Put the matching skills near the top of the resume.")
        elif s < 35:
            tips.append("This role may be a stretch right now. Look at roles closer to your "
                        "current skills (see 'Best-fit roles'), or build the missing skills first.")
    failed = [c for c in ats["checks"] if c["status"] != "pass"]
    failed.sort(key=lambda c: c["max"] - c["points"], reverse=True)
    for c in failed[:4]:
        tips.append(c["tip"])
    if profile["word_count"] < 100:
        tips.insert(0, "Very little text was read from this file. If it is a scanned/image PDF, "
                       "export a text-based PDF or DOCX instead.")
    return tips

In [ ]:
%%writefile resume_ai/predictor.py
"""Load the trained job-category model and predict the best-fit roles for a resume."""
from __future__ import annotations

import threading
from typing import List, Optional

import joblib

from .config import MODELS_DIR
from .preprocessing import clean_text
from .skills import match_skills

_lock = threading.Lock()
_cache: dict = {}


def models_available() -> bool:
    return (MODELS_DIR / "tfidf.pkl").exists() and (MODELS_DIR / "model.pkl").exists()


def _load():
    with _lock:
        if not _cache:
            if not models_available():
                raise FileNotFoundError("Model files not found. Run `python train.py` first.")
            _cache["tfidf"] = joblib.load(MODELS_DIR / "tfidf.pkl")
            _cache["model"] = joblib.load(MODELS_DIR / "model.pkl")
    return _cache["tfidf"], _cache["model"]


def predict_roles(text: str, top_n: int = 3, cleaned: Optional[str] = None, with_skills: bool = True) -> List[dict]:
    """[{category, confidence, matched_skills, missing_skills}, ...] best first."""
    tfidf, model = _load()
    vec = tfidf.transform([cleaned if cleaned is not None else clean_text(text)])
    classes = list(model.classes_)
    if hasattr(model, "predict_proba"):
        ranked = sorted(zip(classes, model.predict_proba(vec)[0]), key=lambda x: x[1], reverse=True)[:top_n]
    else:
        ranked = [(model.predict(vec)[0], 1.0)]
    out = []
    for cat, prob in ranked:
        matched, missing = match_skills(text, cat) if with_skills else ([], [])
        out.append({"category": cat, "confidence": round(float(prob) * 100, 1),
                    "matched_skills": matched, "missing_skills": missing})
    return out

In [ ]:
%%writefile resume_ai/analyzer.py
"""Everything we say about ONE resume (optionally against a prepared job)."""
from __future__ import annotations

from typing import Dict, Optional

from .advice import build_advice
from .ats import ats_check
from .extractor import build_profile
from .matcher import JobSpec, match, prepare_job
from .predictor import models_available, predict_roles
from .preprocessing import clean_text
from .skills import extract_skills


def analyze_resume(text: str, job: Optional[JobSpec] = None, weights: Optional[Dict[str, float]] = None,
                   sem: Optional[float] = None, roles: bool = True) -> dict:
    clean = clean_text(text)
    profile = build_profile(text)
    ats = ats_check(text)
    m = match(text, job, weights, sem, resume_clean=clean) if job is not None else None
    role_list = None
    if roles and models_available():
        try:
            role_list = predict_roles(text, 3, cleaned=clean)
        except Exception:
            role_list = None
    return {
        "profile": profile, "skills_found": extract_skills(text), "ats": ats, "match": m,
        "roles": role_list, "advice": build_advice(profile, ats, m),
    }


def analyze(resume_text: str, jd_text: str = "") -> dict:
    """Convenience wrapper: one CV, optional job description given as text."""
    job = prepare_job(jd_text.strip()) if jd_text and jd_text.strip() else None
    return analyze_resume(resume_text, job)

### 2c. Background job, exports and the Streamlit page

In [ ]:
%%writefile resume_ai/exporters.py
"""Table rows, CSV / Markdown reports, ZIP bundles and saved runs."""
from __future__ import annotations

import csv
import io
import json
import re
import zipfile
from typing import Dict, List, Optional

from .config import OUT, REVIEW_AT, SHORTLIST_AT


def safe(name: str) -> str:
    return re.sub(r"[^\w.\-() ]", "_", name) or "document"


def suggestion(score: Optional[float], shortlist: float = SHORTLIST_AT, review: float = REVIEW_AT) -> str:
    if score is None:
        return "-"
    return "Shortlist" if score >= shortlist else "Review" if score >= review else "Low match"


def rows(results: Dict[str, dict], shortlist: float = SHORTLIST_AT, review: float = REVIEW_AT) -> List[dict]:
    """One flat row per CV, best match first (or by file name when there is no job description)."""
    out = []
    for n, r in results.items():
        m, p = r["match"], r["profile"]
        score = m["score"] if m else None
        out.append({
            "name": n, "alias": r["alias"], "score": score,
            "verdict": m["verdict"] if m else "-", "suggestion": suggestion(score, shortlist, review),
            "skills_matched": (len(m["matched_required"]) + len(m["matched_preferred"])) if m else None,
            "skills_total": (len(m["required_skills"]) + len(m["preferred_skills"])) if m else None,
            "matched_required": m["matched_required"] if m else [],
            "missing_required": m["missing_required"] if m else [],
            "years": p["years_experience"], "education": p["education_label"] or "-",
            "ats": r["ats"]["score"], "role": r["roles"][0]["category"] if r.get("roles") else "-",
            "n_skills": len(r["skills_found"]), "note": r.get("note", ""),
        })
    out.sort(key=lambda x: (-(x["score"] if x["score"] is not None else -1), x["name"].lower()))
    for i, x in enumerate(out, 1):
        x["rank"] = i
    return out


def ranking_csv(rs: List[dict], blind: bool = False) -> str:
    buf = io.StringIO()
    w = csv.writer(buf)
    w.writerow(["Rank", "CV", "Match %", "Verdict", "Suggestion", "Skills matched", "Skills asked",
                "Missing required skills", "Years experience", "Education", "ATS score", "Best-fit role", "Note"])
    for r in rs:
        w.writerow([r["rank"], r["alias"] if blind else r["name"], r["score"] if r["score"] is not None else "",
                    r["verdict"], r["suggestion"], r["skills_matched"] if r["skills_matched"] is not None else "",
                    r["skills_total"] if r["skills_total"] is not None else "", "; ".join(r["missing_required"]),
                    r["years"], r["education"], r["ats"], r["role"], r["note"]])
    return buf.getvalue()


def to_markdown(r: dict, title: Optional[str] = None) -> str:
    """A readable one-CV report."""
    m, p = r["match"], r["profile"]
    L = [f"# {title or r['name']}", ""]
    if m:
        L += [f"**Match: {m['score']:.0f}% - {m['verdict']}**", "", "| Part | Score | Weight | Detail |", "|---|---|---|---|"]
        L += [f"| {b['name']} | {b['score']}% | {b['weight']}% | {b['detail']} |" for b in m["breakdown"] if b["score"] is not None]
        L += ["", f"- Skills you have: {', '.join(m['matched_required'] + m['matched_preferred']) or 'none'}",
              f"- Required skills missing: {', '.join(m['missing_required']) or 'none'}",
              f"- Preferred skills missing: {', '.join(m['missing_preferred']) or 'none'}", ""]
    L += [f"**Read from CV:** {p['years_experience']:g} years experience, education: {p['education_label'] or 'not found'}, "
          f"{p['word_count']} words, {len(r['skills_found'])} skills", "",
          f"**ATS readability:** {r['ats']['score']}/100", ""]
    L += [f"- {c['name']}: {c['status']}" + (f" - {c['tip']}" if c["status"] != "pass" else "") for c in r["ats"]["checks"]]
    L += ["", "## What to do next", ""] + [f"{i}. {t}" for i, t in enumerate(r["advice"], 1)]
    if r.get("roles"):
        L += ["", "## Best-fit roles", ""] + [f"- {x['category']} ({x['confidence']}%)" for x in r["roles"]]
    return "\n".join(L) + "\n"


def build_zip(results: Dict[str, dict], rs: List[dict], blind: bool = False) -> bytes:
    buf = io.BytesIO()
    with zipfile.ZipFile(buf, "w", zipfile.ZIP_DEFLATED) as z:
        z.writestr("ranking.csv", ranking_csv(rs, blind))
        for n, r in results.items():
            label = r["alias"] if blind else n
            z.writestr(f"reports/{safe(label)}.md", to_markdown(r, label))
    return buf.getvalue()


def load_last_run() -> Optional[dict]:
    files = sorted(OUT.glob("*/results.json")) if OUT.exists() else []
    try:
        return json.loads(files[-1].read_text(encoding="utf-8")) if files else None
    except Exception:
        return None

In [ ]:
%%writefile resume_ai/job.py
"""Runs a whole screening in a BACKGROUND THREAD; the page only reads the job's state
(so reloading the browser never loses a run)."""
from __future__ import annotations

import hashlib
import json
import re
import threading
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
from typing import Dict, List, Optional, Tuple

from . import exporters
from .analyzer import analyze_resume
from .config import OUT, READ_WORKERS
from .matcher import JobSpec, prepare_job
from .predictor import models_available, predict_roles
from .readers import ReadError, read_file

WEIGHT = {"queued": 0.0, "reading": 0.15, "analysing": 0.6, "done": 1.0, "failed": 1.0, "skipped": 1.0}
LABEL = {"queued": "⏳ Queued", "reading": "📖 Reading", "analysing": "🧭 Analysing", "done": "✅ Done",
         "failed": "❌ Failed", "skipped": "↪ Duplicate"}
COLOR = {"queued": "#cbd5e1", "reading": "#38bdf8", "analysing": "#6366f1", "done": "#10b981",
         "failed": "#ef4444", "skipped": "#f59e0b"}
BUSY = ("reading", "analysing")


class Job:
    def __init__(self, items: List[Tuple[str, bytes]], jd_text: str, cfg: dict, embedder=None):
        self.items = dict(items)                         # file name -> bytes (released as soon as it is read)
        self.jd_text, self.cfg, self.embedder = (jd_text or "").strip(), cfg, embedder
        self.files = {n: dict(name=n, status="queued", words=0, score=None, sec=0.0, error="")
                      for n in self.items}
        self.aliases = {n: f"Candidate {i:03d}" for i, n in enumerate(self.items, 1)}
        self.results: Dict[str, dict] = {}
        self.timing: Dict[str, float] = {}
        self.meta: dict = {}
        self.phase, self.frac, self.error = "Starting ...", 0.0, ""
        self.t0, self.t1, self.out_dir, self.zip = time.time(), None, "", b""
        self.cancel, self.lock = threading.Event(), threading.RLock()
        threading.Thread(target=self._main, daemon=True, name="cv-job").start()

    done = property(lambda self: self.t1 is not None)
    elapsed = property(lambda self: (self.t1 or time.time()) - self.t0)

    def upd(self, name: str, **kw) -> None:
        with self.lock:
            self.files[name].update(kw)

    def say(self, text: str, frac: float) -> None:
        self.phase, self.frac = text, max(self.frac, min(frac, 0.99))

    def snapshot(self) -> List[dict]:
        with self.lock:
            return [dict(f) for f in self.files.values()]

    # ------------------------------------------------------------------
    def _main(self) -> None:
        try:
            self._run()
            if self.results:
                self._save()
        except Exception as e:
            self.error = f"{type(e).__name__}: {e}"
            for n, f in self.files.items():
                if f["status"] not in ("done", "failed", "skipped"):
                    self.upd(n, status="failed", error=self.error)
        finally:
            self.t1, self.phase, self.frac = time.time(), "Done", 1.0

    def _read_one(self, name: str, ocr: bool):
        if self.cancel.is_set():
            return name, None, "stopped by the user"
        self.upd(name, status="reading")
        try:
            r = read_file(name, self.items[name], ocr=ocr)
            return name, r, ""
        except ReadError as e:
            return name, None, str(e)
        except Exception as e:
            return name, None, f"{type(e).__name__}: {e}"
        finally:
            self.items[name] = b""                        # free the memory

    def _run(self) -> None:
        cfg, t = self.cfg, time.time()
        spec: Optional[JobSpec] = prepare_job(self.jd_text) if self.jd_text else None
        if spec and cfg.get("roles", True) and models_available():
            try:
                spec.role = predict_roles(self.jd_text, 1, with_skills=False)[0]["category"]
            except Exception:
                pass
        self.meta = dict(
            has_job=spec is not None, required=spec.required if spec else [], preferred=spec.preferred if spec else [],
            need_years=spec.need_years if spec else None, need_education=spec.need_label if spec else None,
            job_role=spec.role if spec else None, weights=cfg.get("weights"),
            shortlist=cfg.get("shortlist"), review=cfg.get("review"),
            semantic=bool(self.embedder and getattr(self.embedder, "model", None)),
        )

        # ---- 1. read all files in parallel
        self.say(f"Reading {len(self.items)} files ...", 0.02)
        texts: Dict[str, str] = {}
        notes: Dict[str, str] = {}
        kinds: Dict[str, str] = {}
        finished = 0
        with ThreadPoolExecutor(max_workers=min(READ_WORKERS, max(1, len(self.items)))) as pool:
            futs = [pool.submit(self._read_one, n, cfg.get("ocr", True)) for n in list(self.items)]
            for fu in as_completed(futs):
                n, r, err = fu.result()
                finished += 1
                self.say(f"Reading files ... {finished}/{len(self.items)}", 0.02 + 0.4 * finished / len(self.items))
                if r is None:
                    self.upd(n, status="failed", error=err)
                    continue
                texts[n], notes[n], kinds[n] = r.text, r.warning, r.kind
                self.upd(n, status="analysing", words=len(r.text.split()))
        self.timing["read"] = time.time() - t
        if self.cancel.is_set():
            for n, f in self.files.items():
                if f["status"] not in ("done", "failed", "skipped"):
                    self.upd(n, status="failed", error="stopped by the user")
            return

        # keep the original upload order, drop exact duplicates
        order = [n for n in self.items if n in texts]
        seen: Dict[str, str] = {}
        for n in order:
            key = hashlib.md5(re.sub(r"\W+", "", texts[n].lower()).encode()).hexdigest()
            if key in seen:
                self.upd(n, status="skipped", error=f"Same content as {seen[key]}")
                texts.pop(n)
            else:
                seen[key] = n
        order = [n for n in order if n in texts]
        if not order:
            return

        # ---- 2. optional meaning-based similarity, all CVs in one batch
        sims: Dict[str, float] = {}
        if spec and self.embedder is not None and getattr(self.embedder, "model", None) is not None:
            t = time.time()
            self.say(f"Embedding {len(order)} CVs ({self.embedder.name}) ...", 0.45)
            E = self.embedder.encode_docs([self.jd_text] + [texts[n] for n in order])
            if E is not None:
                sims = {n: float(E[i + 1] @ E[0]) for i, n in enumerate(order)}
            self.timing["embed"] = time.time() - t

        # ---- 3. analyse every CV
        t = time.time()
        for k, n in enumerate(order):
            if self.cancel.is_set():
                break
            self.say(f"Analysing {n} ({k + 1}/{len(order)}) ...", 0.5 + 0.45 * k / len(order))
            t1 = time.time()
            try:
                res = analyze_resume(texts[n], spec, cfg.get("weights"), sims.get(n), cfg.get("roles", True))
            except Exception as e:
                self.upd(n, status="failed", error=f"{type(e).__name__}: {e}")
                continue
            res.update(name=n, alias=self.aliases[n], kind=kinds[n], note=notes[n], words=len(texts[n].split()))
            self.results[n] = res
            self.upd(n, status="done", sec=time.time() - t1, error=notes[n],
                     score=res["match"]["score"] if res["match"] else None)
        self.timing["analyse"] = time.time() - t
        if self.cancel.is_set():
            for n, f in self.files.items():
                if f["status"] not in ("done", "failed", "skipped"):
                    self.upd(n, status="failed", error="stopped by the user")
        self.timing["total"] = sum(self.timing.values())

    def _save(self) -> None:
        d = OUT / time.strftime("%Y%m%d-%H%M%S")
        d.mkdir(parents=True, exist_ok=True)
        self.out_dir = str(d)
        rs = exporters.rows(self.results, self.cfg.get("shortlist", 65), self.cfg.get("review", 40))
        (d / "ranking.csv").write_text(exporters.ranking_csv(rs), encoding="utf-8")
        self.zip = exporters.build_zip(self.results, rs)
        (d / "reports.zip").write_bytes(self.zip)
        meta = {**self.meta, "seconds": round(self.elapsed, 1), "n_files": len(self.files)}
        (d / "results.json").write_text(json.dumps({"meta": meta, "results": self.results}), encoding="utf-8")

In [ ]:
%%writefile resume_ai/ui.py
"""The Streamlit page: sidebar, job + upload panel, live progress and results.

Everything - the job description, the CV files, the settings - is entered in the browser.
There is nothing to edit or upload in code.
"""
from __future__ import annotations

import html
import json
import shutil
from pathlib import Path
from typing import Dict, List, Optional

import streamlit as st

from . import exporters, semantic
from .config import MAX_DOCS, MAX_FILE_MB, REVIEW_AT, ROOT, SHORTLIST_AT, UPLOAD_TYPES, WEIGHTS
from .exporters import safe
from .job import BUSY, COLOR, LABEL, WEIGHT, Job
from .predictor import models_available
from .readers import ReadError, expand_zip

CSS = """<style>
.block-container{padding-top:1.3rem;max-width:1650px}
header[data-testid="stHeader"]{background:transparent}
.hero{display:flex;justify-content:space-between;align-items:center;gap:1rem;flex-wrap:wrap;padding:1.35rem 1.8rem;border-radius:1.1rem;color:#fff;background:linear-gradient(120deg,#0f766e 0%,#4f46e5 55%,#0ea5e9 100%);box-shadow:0 10px 28px rgba(79,70,229,.25);margin-bottom:1.1rem}
.hero-t{font-size:1.85rem;font-weight:800;letter-spacing:-.02em;line-height:1.2}.hero-s{opacity:.92;margin-top:.25rem;font-size:.95rem}
.chips{display:flex;gap:.5rem;flex-wrap:wrap}.chip{background:rgba(255,255,255,.16);border:1px solid rgba(255,255,255,.3);padding:.3rem .85rem;border-radius:2rem;font-size:.8rem;font-weight:600}
.kpis{display:grid;grid-template-columns:repeat(4,minmax(0,1fr));gap:.75rem;margin:.35rem 0 .8rem}
.kpi{background:#fff;border:1px solid #e5e7eb;border-top:3px solid var(--accent,#4f46e5);border-radius:.8rem;padding:.65rem .95rem;box-shadow:0 1px 3px rgba(15,23,42,.06)}
.kpi-l{font-size:.78rem;color:#64748b;font-weight:700}.kpi-v{font-size:1.5rem;font-weight:800;color:#0f172a;line-height:1.3}.kpi-s{font-size:.78rem;color:#64748b}
.sec{font-size:.9rem;color:#475569;font-weight:700;margin:1rem 0 .4rem}
.tiles{display:flex;flex-wrap:wrap;gap:.3rem}.tile{width:2.15rem;height:2.15rem;border-radius:.55rem;color:#fff;font-size:.7rem;font-weight:700;display:flex;align-items:center;justify-content:center}
.tile.busy{animation:pulse 1.2s ease-in-out infinite}@keyframes pulse{50%{opacity:.55}}
.legend{display:flex;flex-wrap:wrap;gap:.9rem;margin-top:.45rem;font-size:.75rem;color:#64748b}.legend i{display:inline-block;width:.7rem;height:.7rem;border-radius:.2rem;margin-right:.3rem}
.shero{display:flex;gap:1.6rem;align-items:center;background:#fff;border:1px solid #e5e7eb;border-left:8px solid var(--c);border-radius:.8rem;padding:1.1rem 1.5rem;margin:.4rem 0 .8rem;color:#0f172a}
.shero.good{--c:#0e8a5f}.shero.ok{--c:#6b8e23}.shero.mid{--c:#b87a0a}.shero.low{--c:#c2410c}
.snum{font-size:4rem;font-weight:800;line-height:1;color:var(--c);letter-spacing:-.04em}.snum small{font-size:1.6rem;margin-left:2px}
.sver{font-size:1.35rem;font-weight:800}.ssub{color:#475569;margin-top:.15rem}
.strip{display:flex;gap:6px;margin:.2rem 0 .6rem}.seg{min-width:105px}.seg-bar{height:13px;background:#dfe4ec;border-radius:3px;overflow:hidden}
.seg-fill{height:100%;background:var(--c)}.seg-fill.good{--c:#0e8a5f}.seg-fill.ok{--c:#6b8e23}.seg-fill.mid{--c:#b87a0a}.seg-fill.low{--c:#c2410c}
.seg-name{font-size:.84rem;margin-top:5px;color:#0f172a}.seg-detail{font-size:.74rem;color:#64748b;line-height:1.3}
.srow{display:grid;grid-template-columns:140px 1fr;gap:10px;padding:.5rem 0;border-top:1px solid #eceff3;align-items:start}.srow:first-of-type{border-top:0}
.slbl{font-size:.84rem;font-weight:700;color:#475569;padding-top:3px}
.sk{display:inline-block;font-size:.83rem;padding:2px 10px;margin:0 6px 6px 0;border-radius:99px;border:1px solid transparent}
.sk.have{background:#e3f4ec;color:#0a6444;border-color:#b5dfcb}.sk.need{background:#fdebe3;color:#9a3412;border-color:#f2c4ae}
.sk.nice{background:#fff4d6;color:#8a5a00;border-color:#efd88f}.sk.extra{background:#eef1f5;color:#475569;border-color:#dde2ea}
.sk.kw{background:#fff;color:#0f172a;border-color:#b9c1ce;border-style:dashed}
@media (max-width:1100px){.kpis{grid-template-columns:repeat(2,minmax(0,1fr))}.strip{flex-direction:column}.seg{flex:none!important}}
</style>"""

TYPE_HELP = "PDF, DOCX, DOC, RTF, ODT, TXT, HTML, scanned images (JPG/PNG), or a ZIP full of CVs"


# ---------------------------------------------------------------- shared state / helpers
@st.cache_resource(show_spinner=False)
def registry() -> dict:
    return {"job": None, "loaded": None}              # shared by every browser tab, survives page reloads


@st.cache_resource(show_spinner=False)
def get_embedder(name: str):
    return semantic.Embedder(name)


def fmt_dur(sec: float) -> str:
    sec = int(round(max(sec or 0.0, 0.0)))
    h, rem = divmod(sec, 3600)
    return f"{h}:{rem // 60:02d}:{rem % 60:02d}" if h else f"{rem // 60}:{rem % 60:02d}"


def show_table(rows, **kw) -> None:
    try:
        st.dataframe(rows, width="stretch", hide_index=True, **kw)
    except Exception:
        st.dataframe(rows, use_container_width=True, hide_index=True, **kw)


def kpi(label: str, value: str, sub: str = "", accent: str = "#4f46e5") -> str:
    return (f'<div class="kpi" style="--accent:{accent}"><div class="kpi-l">{html.escape(label)}</div>'
            f'<div class="kpi-v">{html.escape(value)}</div><div class="kpi-s">{html.escape(sub)}</div></div>')


def kpi_row(*cards: str) -> None:
    st.markdown('<div class="kpis">' + "".join(cards) + "</div>", unsafe_allow_html=True)


def level(score: float) -> str:
    return "good" if score >= 75 else "ok" if score >= 55 else "mid" if score >= 35 else "low"


def chips(items: List[str], cls: str, empty: str = "none") -> str:
    if not items:
        return f'<span style="color:#64748b;font-size:.85rem">{empty}</span>'
    return "".join(f'<span class="sk {cls}">{html.escape(i)}</span>' for i in items)


def srow(label: str, inner: str) -> str:
    return f'<div class="srow"><div class="slbl">{html.escape(label)}</div><div>{inner}</div></div>'


def tiles_html(files: List[dict]) -> str:
    tiles = "".join(f'<div class="tile{" busy" if f["status"] in BUSY else ""}" style="background:{COLOR[f["status"]]}" '
                    f'title="{html.escape(f["name"])} - {LABEL[f["status"]][2:]}">{i}</div>' for i, f in enumerate(files, 1))
    legend = "".join(f'<span><i style="background:{COLOR[s]}"></i>{LABEL[s].split(" ", 1)[1]}</span>' for s in COLOR)
    return f'<div class="tiles">{tiles}</div><div class="legend">{legend}</div>'


# ---------------------------------------------------------------- live progress
def live_panel(watching: bool) -> None:
    job: Optional[Job] = registry()["job"]
    if job is None:
        st.info("Progress for every CV appears here when you press **Analyze**.")
        return
    files = job.snapshot()
    n_done = sum(f["status"] == "done" for f in files)
    n_bad = sum(f["status"] in ("failed", "skipped") for f in files)
    finished = n_done + n_bad
    st.progress(min(max(job.frac, 0.0), 1.0), text=f"{job.phase}  ·  {finished}/{len(files)} files  ·  {fmt_dur(job.elapsed)} elapsed")
    if job.error:
        st.error(f"The run stopped unexpectedly: {job.error}")
    rate = f"{60 * n_done / job.elapsed:,.0f} CVs/min" if job.elapsed > 0 and n_done else "-"
    kpi_row(kpi("Files finished", f"{finished} / {len(files)}", f"{n_done} analysed"),
            kpi("Elapsed time", fmt_dur(job.elapsed), "running" if not job.done else "finished", "#0ea5e9"),
            kpi("Speed", rate, "analysed CVs only", "#10b981"),
            kpi("Problems", str(n_bad), "unreadable or duplicate files", "#e11d48" if n_bad else "#94a3b8"))
    st.markdown('<div class="sec">Files</div>' + tiles_html(files), unsafe_allow_html=True)
    cc = st.column_config
    show_table([{"#": i, "File": f["name"], "Status": LABEL[f["status"]], "Progress": round(100 * WEIGHT[f["status"]]),
                 "Words": f["words"] or None, "Match %": f["score"], "Time (s)": round(f["sec"], 2) or None, "Note": f["error"]}
                for i, f in enumerate(files, 1)],
               column_config={"Progress": cc.ProgressColumn("Progress", min_value=0, max_value=100, format="%d%%"),
                              "Match %": cc.NumberColumn("Match %", format="%.0f")})
    if job.timing:
        with st.expander("Where the time went"):
            show_table([{"Step": k, "Seconds": round(v, 2)} for k, v in job.timing.items()])
    if watching and job.done:
        st.rerun()


# ---------------------------------------------------------------- results
def render_cv(r: dict, meta: dict, title: str) -> None:
    m, p = r["match"], r["profile"]
    if r.get("note"):
        st.info(r["note"])
    if m:
        miss = len(m["missing_required"])
        sub = (f"Missing {miss} of the {len(m['required_skills'])} required skills." if miss else
               "Every required skill in the posting appears in this CV." if m["required_skills"] else
               "No specific skills were recognised in the posting, so wording and experience carry the score.")
        st.markdown(f'<div class="shero {level(m["score"])}"><div class="snum">{m["score"]:.0f}<small>%</small></div>'
                    f'<div><div class="sver">{html.escape(m["verdict"])}</div><div class="ssub">{html.escape(sub)}</div></div></div>',
                    unsafe_allow_html=True)
        segs = "".join(
            f'<div class="seg" style="flex:{b["weight"]}"><div class="seg-bar"><div class="seg-fill {level(b["score"])}" style="width:{b["score"]}%"></div></div>'
            f'<div class="seg-name">{html.escape(b["name"])} <b>{b["score"]}%</b></div>'
            f'<div class="seg-detail">{html.escape(b["detail"])} · counts {b["weight"]}%</div></div>'
            for b in m["breakdown"] if b["score"] is not None)
        st.markdown(f'<div class="strip">{segs}</div>', unsafe_allow_html=True)
        if meta.get("job_role") and r.get("roles"):
            best = r["roles"][0]["category"]
            same = "Same field - good sign." if best == meta["job_role"] else "Different fields - expect a harder fit."
            st.caption(f"This job reads like **{meta['job_role']}**; this CV reads most like **{best}**. {same}")
        st.markdown('<div class="sec">Skills the job asks for</div>' +
                    srow("CV has", chips(m["matched_required"] + m["matched_preferred"], "have")) +
                    srow("Required, missing", chips(m["missing_required"], "need", "nothing missing")) +
                    srow("Preferred, missing", chips(m["missing_preferred"], "nice", "nothing missing")) +
                    (srow("Also on the CV", chips(m["extra_skills"], "extra")) if m["extra_skills"] else "") +
                    (srow("Words to consider", chips(m["missing_keywords"][:10], "kw")) if m["missing_keywords"] else ""),
                    unsafe_allow_html=True)
    else:
        st.info("No job description was given, so there is no match score. Profile, ATS check and role fit are shown below.")

    kpi_row(kpi("Experience", f"{p['years_experience']:g} yrs", "found in the CV", "#8b5cf6"),
            kpi("Education", p["education_label"] or "not found", "highest level", "#e11d48"),
            kpi("Skills found", str(len(r["skills_found"])), "from the skill vocabulary", "#0ea5e9"),
            kpi("ATS readability", f"{r['ats']['score']} / 100", f"{p['word_count']} words", "#10b981"))

    st.markdown('<div class="sec">What to do next</div>', unsafe_allow_html=True)
    st.markdown("\n".join(f"{i}. {t}" for i, t in enumerate(r["advice"], 1)) or "Nothing urgent - this CV reads well.")

    with st.expander("ATS readability checks"):
        icon = {"pass": "✅ Pass", "partial": "🟡 Partly", "fail": "❌ Fix"}
        show_table([{"Check": c["name"], "Result": icon[c["status"]], "Points": f"{c['points']}/{c['max']}",
                     "Tip": c["tip"] if c["status"] != "pass" else ""} for c in r["ats"]["checks"]])
    with st.expander("All skills found in this CV"):
        st.markdown(chips(r["skills_found"], "extra"), unsafe_allow_html=True)
    if r.get("roles"):
        with st.expander("Best-fit roles (ML model)"):
            show_table([{"Role": x["category"], "Confidence %": x["confidence"], "Skills it has": ", ".join(x["matched_skills"]),
                         "Skills to build": ", ".join(x["missing_skills"])} for x in r["roles"]])
    st.download_button("⬇ This CV's report (.md)", exporters.to_markdown(r, title), f"{safe(title)}.md", "text/markdown",
                       key=f"md_{safe(title)}")


def render_results(ds: dict) -> None:
    reg = registry()
    job: Optional[Job] = reg["job"]
    if job is not None and job.done and job.results:
        results, meta = job.results, {**job.meta, "seconds": job.elapsed}
    elif job is None and reg["loaded"]:
        results, meta = reg["loaded"]["results"], reg["loaded"]["meta"]
    else:
        st.info("Results appear here when the run finishes: a ranked table, filters, and a full report for every CV.")
        return

    blind, has_job = ds["blind"], meta.get("has_job")
    allrows = exporters.rows(results, ds["shortlist"], ds["review"])
    disp = (lambda r: r["alias"]) if blind else (lambda r: r["name"])
    n_sl = sum(r["suggestion"] == "Shortlist" for r in allrows)
    n_rv = sum(r["suggestion"] == "Review" for r in allrows)
    avg = sum(r["score"] for r in allrows) / len(allrows) if has_job else None
    kpi_row(kpi("CVs analysed", str(len(allrows)), f"in {fmt_dur(meta.get('seconds', 0))}"),
            kpi("Shortlist", str(n_sl) if has_job else "-", f"match ≥ {ds['shortlist']}%", "#10b981"),
            kpi("Review", str(n_rv) if has_job else "-", f"match {ds['review']}–{ds['shortlist']}%", "#d97706"),
            kpi("Average match", f"{avg:.0f}%" if avg is not None else "-", "all analysed CVs", "#0ea5e9"))

    if has_job:
        bits = [f"Required skills read from the job: **{', '.join(meta['required']) or 'none recognised'}**"]
        if meta.get("preferred"):
            bits.append(f"Preferred: {', '.join(meta['preferred'])}")
        if meta.get("need_years") is not None:
            bits.append(f"Experience asked: {meta['need_years']:g}+ yrs")
        st.caption(" · ".join(bits))
    if blind:
        st.caption("🕶️ Blind screening is on - CVs are shown as Candidate numbers. Turn it off in the sidebar to see file names.")

    # ---- filters
    shown = allrows
    if has_job:
        c1, c2 = st.columns([1.2, 1.2])
        min_score = c1.slider("Minimum match %", 0, 100, 0)
        keep = c2.multiselect("Show", ["Shortlist", "Review", "Low match"], default=["Shortlist", "Review", "Low match"])
        shown = [r for r in shown if r["score"] >= min_score and r["suggestion"] in keep]
        if meta.get("required"):
            must = st.multiselect("Only CVs that have ALL of these required skills", meta["required"])
            shown = [r for r in shown if all(s in r["matched_required"] for s in must)]
    q = st.text_input("Search by name", placeholder="type part of a file name" if not blind else "search disabled in blind mode",
                      disabled=blind).strip().lower()
    if q:
        shown = [r for r in shown if q in r["name"].lower()]

    cc = st.column_config
    st.caption(f"Showing {len(shown)} of {len(allrows)} CVs")
    show_table([{"#": r["rank"], "CV": disp(r), "Match %": r["score"], "Suggestion": r["suggestion"],
                 "Skills": f"{r['skills_matched']}/{r['skills_total']}" if has_job else None,
                 "Missing required": ", ".join(r["missing_required"]), "Years": r["years"], "Education": r["education"],
                 "ATS": r["ats"], "Best-fit role": r["role"], "Note": r["note"]} for r in shown],
               column_config={"Match %": cc.ProgressColumn("Match %", min_value=0, max_value=100, format="%.0f%%"),
                              "ATS": cc.ProgressColumn("ATS", min_value=0, max_value=100, format="%d")})
    d1, d2 = st.columns(2)
    d1.download_button(f"⬇ This table ({len(shown)} CVs, .csv)", exporters.ranking_csv(shown, blind), "ranking.csv", "text/csv",
                       disabled=not shown)
    d2.download_button(f"📦 All {len(allrows)} reports + ranking (.zip)", exporters.build_zip(results, allrows, blind),
                       "cv_reports.zip", "application/zip")

    if shown:
        st.markdown('<div class="sec">Open one CV</div>', unsafe_allow_html=True)
        names = {disp(r): r["name"] for r in shown}
        pick = st.selectbox("CV", list(names), label_visibility="collapsed")
        render_cv(results[names[pick]], meta, pick)


# ---------------------------------------------------------------- sidebar + upload
def sidebar() -> dict:
    with st.sidebar:
        st.markdown("### ⚙️ Settings")
        with st.expander("Match score weights"):
            st.caption("How much each part counts. Parts a job does not mention are skipped automatically.")
            w = {"skills": st.slider("Skills", 0, 100, int(WEIGHTS["skills"] * 100)),
                 "text": st.slider("Text similarity", 0, 100, int(WEIGHTS["text"] * 100)),
                 "experience": st.slider("Experience", 0, 100, int(WEIGHTS["experience"] * 100)),
                 "education": st.slider("Education", 0, 100, int(WEIGHTS["education"] * 100))}
            if sum(w.values()) == 0:
                w = {k: v * 100 for k, v in WEIGHTS.items()}
        sem = st.toggle("Meaning-based similarity", value=False, disabled=not semantic.available(),
                        help="Understands that 'built ML models' ≈ 'machine learning engineer'. Needs sentence-transformers "
                             "(downloads ~90 MB the first time).")
        if not semantic.available():
            st.caption("Install `sentence-transformers` to enable this.")
        roles = st.toggle("Suggest best-fit role", value=models_available(), disabled=not models_available(),
                          help="Uses the trained model in models/. Run train.py to create it.")
        has_ocr = shutil.which("tesseract") is not None
        ocr = st.toggle("Read scanned CVs (OCR)", value=has_ocr, disabled=not has_ocr,
                        help="Only used for image files and PDFs without text. Slower: a few seconds per page.")
        if not has_ocr:
            st.caption("Install `tesseract-ocr` to read scanned CVs.")
        st.divider()
        shortlist = st.slider("Shortlist from match %", 30, 95, SHORTLIST_AT)
        review = min(st.slider("Review from match %", 5, 90, REVIEW_AT), shortlist)
        blind = st.toggle("Blind screening", value=False,
                          help="Show Candidate 001, 002 ... instead of file names (names often contain the person's name).")
        st.divider()
        if st.button("Load last saved run"):
            data = exporters.load_last_run()
            if data:
                registry().update(job=None, loaded=data)
                st.rerun()
            st.warning("No saved run found.")
        st.caption(f"Up to **{MAX_DOCS}** CVs per run · {MAX_FILE_MB} MB per file")
    return dict(weights={k: v / 100 for k, v in w.items()}, semantic=sem, roles=roles, ocr=ocr,
                shortlist=shortlist, review=review, blind=blind)


def _unique(name: str, taken: set) -> str:
    base, stem, i = name, Path(name).stem, 2
    suffix = Path(name).suffix
    while name in taken:
        name, i = f"{stem} ({i}){suffix}", i + 1
    return name


def start_job(files, jd: str, cfg: dict) -> None:
    items, taken, problems = [], set(), []
    with st.status("Unpacking files ...", expanded=False) as status:
        for f in files:
            data = f.getvalue()
            if f.name.lower().endswith(".zip"):
                try:
                    inner = expand_zip(data, MAX_DOCS + 1 - len(items))
                    problems.append(f"{f.name}: {len(inner)} files inside")
                except ReadError as e:
                    problems.append(f"{f.name}: {e}")
                    continue
                batch = inner
            else:
                batch = [(f.name, data)]
            for n, d in batch:
                n = _unique(n, taken)
                taken.add(n)
                items.append((n, d))
        status.update(label=f"{len(items)} files ready", state="complete")
    for p in problems:
        st.caption(p)
    if not items:
        st.error("No readable files found.")
        return
    if len(items) > MAX_DOCS:
        st.error(f"{len(items)} files found - the limit is {MAX_DOCS} per run. Split them into batches "
                 f"(or raise CV_MAX_DOCS).")
        return
    emb = None
    if cfg["semantic"] and jd.strip():
        with st.status("Loading the meaning model ... (first run downloads it)", expanded=False) as status:
            emb = get_embedder(semantic.MODEL)
            if emb.model is None:
                st.warning(f"Meaning model unavailable ({emb.error[:120]}); using word-based similarity only.")
            status.update(label="Model ready", state="complete")
    registry().update(job=Job(items, jd, cfg, emb), loaded=None)
    st.session_state.uploader_key += 1                      # empties the uploader so the files stop using RAM
    st.rerun()


def _example_job() -> None:
    p = ROOT / "samples" / "jd_data_scientist.txt"
    if p.exists():
        st.session_state.jd = p.read_text(encoding="utf-8")


def upload_panel(running: bool, cfg: dict) -> None:
    ss = st.session_state
    ss.setdefault("uploader_key", 0)
    ss.setdefault("jd", "")
    st.subheader("1 · Job description")
    st.text_area("Paste the job posting", key="jd", height=230, disabled=running, label_visibility="collapsed",
                 placeholder="Paste the full job posting here. Leave empty to only profile the CVs (no match score).")
    st.button("Use an example job", on_click=_example_job, disabled=running)

    st.subheader("2 · Upload CVs")
    files = st.file_uploader(f"Drop up to {MAX_DOCS} CVs (or one ZIP of many)", type=UPLOAD_TYPES, accept_multiple_files=True,
                             disabled=running, key=f"up_{ss.uploader_key}") or []
    st.caption(TYPE_HELP)
    if files:
        st.caption(f"{len(files)} selected · {sum(f.size for f in files) / 1e6:.1f} MB")
    go = st.button("🚀 Analyze", type="primary", disabled=not files or running)
    if running:
        job = registry()["job"]
        if st.button("⏹ Stop after the current step"):
            job.cancel.set()
        st.caption("Stopping..." if job.cancel.is_set() else "Running in the background - you can switch tabs or reload the page.")
    if go:
        start_job(files, ss.jd, cfg)
    job = registry()["job"]
    if job is not None and job.done and job.out_dir:
        st.success(f"Finished {len(job.results)} CV(s) in {fmt_dur(job.elapsed)}. Saved in {job.out_dir}")


def main() -> None:
    st.set_page_config(page_title="CV Match Studio", page_icon="🎯", layout="wide", initial_sidebar_state="expanded")
    st.markdown(CSS, unsafe_allow_html=True)
    cfg = sidebar()
    st.markdown('<div class="hero"><div><div class="hero-t">🎯 CV Match Studio</div><div class="hero-s">Rank up to '
                f'{MAX_DOCS} CVs against one job - explainable scores, skill gaps and an ATS check for each CV</div></div>'
                '<div class="chips"><span class="chip">PDF · DOCX · DOC · TXT · RTF · ODT · images · ZIP</span>'
                '<span class="chip">Match %</span><span class="chip">Skill gaps</span><span class="chip">ATS check</span></div></div>',
                unsafe_allow_html=True)
    reg = registry()
    left, right = st.columns([1, 2.3], gap="large")
    with left:
        upload_panel(reg["job"] is not None and not reg["job"].done, cfg)
    job = reg["job"]
    running = job is not None and not job.done
    with right:
        tab_live, tab_res = st.tabs(["📊 Live progress", "📄 Results"])
        with tab_live:
            st.fragment(run_every=1.0 if running else None)(live_panel)(running)
        with tab_res:
            render_results(cfg)

In [ ]:
%%writefile app.py
"""Start the web app:   streamlit run app.py"""
from resume_ai.ui import main

main()

In [ ]:
%%writefile .streamlit/config.toml
[server]
maxUploadSize = 200          # MB per file
maxMessageSize = 500
fileWatcherType = "none"

[browser]
gatherUsageStats = false

In [ ]:
%%writefile train.py
"""Train the resume -> job-category classifier.

Run once:  python train.py

Loads the Kaggle 'Updated Resume Dataset', cleans the text, REMOVES DUPLICATE
RESUMES, builds TF-IDF features, compares Logistic Regression and Naive Bayes with
cross-validation, evaluates the winner on a held-out test set, prints metrics, and
saves the vectorizer + best model + metrics.json to the models/ folder.

Everything runs on CPU in a few seconds.

v2 changes (honest evaluation)
  * The Kaggle file contains many exact duplicate resumes. If a duplicate lands in
    both train and test, accuracy looks like ~99% for the wrong reason. We drop
    duplicates BEFORE splitting, so the reported score reflects unseen resumes.
  * The model is chosen by 5-fold cross-validation on the training data, not by
    peeking at the test set.
"""
import json
import os
import sys

import joblib
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, classification_report,
                             confusion_matrix, f1_score, precision_score,
                             recall_score)
from sklearn.model_selection import (StratifiedKFold, cross_val_score,
                                     train_test_split)
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline

# make the resume_ai package importable
BASE_DIR = os.path.dirname(os.path.abspath(__file__))
sys.path.insert(0, BASE_DIR)
from resume_ai.preprocessing import clean_text  # noqa: E402

DATA_PATH = os.environ.get("RESUME_CSV") or os.path.join(BASE_DIR, "dataset", "UpdatedResumeDataSet.csv")
MODELS_DIR = os.path.join(BASE_DIR, "models")


def make_tfidf():
    return TfidfVectorizer(max_features=1500, stop_words="english", ngram_range=(1, 2))


def load_data():
    if not os.path.exists(DATA_PATH):
        sys.exit(
            f"Dataset not found at {DATA_PATH}\n"
            "Download 'UpdatedResumeDataSet.csv' from Kaggle:\n"
            "  https://www.kaggle.com/datasets/jillanisofttech/updated-resume-dataset\n"
            "and place it in the dataset/ folder."
        )
    df = pd.read_csv(DATA_PATH)

    # be flexible about column names across dataset versions
    text_col = next((c for c in df.columns if "resume" in c.lower()), None)
    cat_col = next((c for c in df.columns if "categor" in c.lower()), None)
    if text_col is None or cat_col is None:
        sys.exit(f"Expected resume/category columns, found: {list(df.columns)}")

    df = df[[cat_col, text_col]].dropna()
    df.columns = ["Category", "Resume"]
    return df


def main():
    print("Loading dataset...")
    df = load_data()
    n_raw = len(df)
    print(f"  {n_raw} resumes across {df['Category'].nunique()} categories")

    # ---- remove duplicates so train and test never share a resume
    df["Resume"] = df["Resume"].astype(str)
    df = df.drop_duplicates(subset="Resume").reset_index(drop=True)
    n_unique = len(df)
    print(f"  Removed {n_raw - n_unique} duplicate resumes -> {n_unique} unique resumes")

    counts = df["Category"].value_counts()
    rare = counts[counts < 4].index.tolist()
    if rare:
        print(f"  Dropping categories with fewer than 4 unique resumes: {rare}")
        df = df[~df["Category"].isin(rare)].reset_index(drop=True)

    print("Cleaning text...")
    df["clean"] = df["Resume"].apply(clean_text)

    X_train, X_test, y_train, y_test = train_test_split(
        df["clean"], df["Category"],
        test_size=0.2, random_state=42, stratify=df["Category"],
    )

    n_splits = max(2, min(5, int(y_train.value_counts().min())))
    cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

    candidates = {
        "Logistic Regression": LogisticRegression(max_iter=1000, C=10),
        "Naive Bayes": MultinomialNB(),
    }

    print(f"\nComparing models ({n_splits}-fold cross-validation on the training set):")
    results, best_name, best_cv = {}, None, -1.0
    for name, clf in candidates.items():
        pipe = Pipeline([("tfidf", make_tfidf()), ("clf", clf)])
        scores = cross_val_score(pipe, X_train, y_train, cv=cv, scoring="f1_weighted")
        results[name] = {"cv_f1_mean": float(scores.mean()), "cv_f1_std": float(scores.std())}
        print(f"  {name:22s}  cv f1 = {scores.mean():.3f} (+/- {scores.std():.3f})")
        if scores.mean() > best_cv:
            best_name, best_cv = name, scores.mean()

    print("\nBuilding final TF-IDF features and fitting all models...")
    tfidf = make_tfidf()
    Xtr = tfidf.fit_transform(X_train)
    Xte = tfidf.transform(X_test)
    fitted = {}
    for name, clf in candidates.items():
        clf.fit(Xtr, y_train)
        preds = clf.predict(Xte)
        results[name]["test_accuracy"] = float(accuracy_score(y_test, preds))
        results[name]["test_f1"] = float(f1_score(y_test, preds, average="weighted"))
        print(f"  {name:22s}  test accuracy={results[name]['test_accuracy']:.3f}  "
              f"f1={results[name]['test_f1']:.3f}")
        fitted[name] = clf

    best_model = fitted[best_name]
    preds = best_model.predict(Xte)
    print(f"\nBest model (by cross-validation): {best_name}")
    print("\nClassification report (held-out test set):")
    print(classification_report(y_test, preds, zero_division=0))
    print("Confusion matrix (rows=true, cols=pred):")
    print(confusion_matrix(y_test, preds))

    metrics = {
        "resumes_raw": n_raw, "resumes_unique": n_unique,
        "categories": int(df["Category"].nunique()),
        "best_model": best_name,
        "accuracy": float(accuracy_score(y_test, preds)),
        "precision": float(precision_score(y_test, preds, average="weighted", zero_division=0)),
        "recall": float(recall_score(y_test, preds, average="weighted", zero_division=0)),
        "f1": float(f1_score(y_test, preds, average="weighted")),
        "models": results,
    }

    os.makedirs(MODELS_DIR, exist_ok=True)
    joblib.dump(tfidf, os.path.join(MODELS_DIR, "tfidf.pkl"))
    joblib.dump(best_model, os.path.join(MODELS_DIR, "model.pkl"))
    with open(os.path.join(MODELS_DIR, "metrics.json"), "w") as fh:
        json.dump(metrics, fh, indent=2)
    print(f"\nSaved tfidf.pkl, model.pkl and metrics.json to {MODELS_DIR}")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile samples/jd_data_scientist.txt
Data Scientist - Retail Analytics

About the role
We are hiring a Data Scientist to build forecasting and recommendation models for our retail business.

Requirements
- 3+ years of experience in data science or analytics
- Bachelor's or Master's degree in a quantitative field
- Strong Python and SQL skills
- Experience with machine learning, scikit-learn and statistics
- Experience deploying models with Docker on AWS
- Excellent communication skills and the ability to work in a team

Nice to have
- Experience with Spark or Kubernetes
- Tableau or Power BI dashboards
- NLP experience

In [ ]:
%%writefile samples/jd_web_developer.txt
Frontend Developer

Requirements
- 2+ years of experience building websites
- Strong HTML, CSS and JavaScript
- Experience with React and TypeScript
- Familiarity with REST API integration and Git

Preferred
- Figma, responsive design
- Unit testing with Jest

In [ ]:
%%writefile samples/resume_data_scientist.txt
Priya Sharma
priya.sharma@example.com | +91 98765 43210 | linkedin.com/in/priyasharma | github.com/priya-ds

SUMMARY
Data scientist with 4 years of experience building machine learning models for retail and fintech.

EXPERIENCE
Data Scientist, RetailCo (2022 - Present)
- Built a demand forecasting model in Python (scikit-learn, XGBoost) that reduced stock-outs by 18%.
- Automated weekly reporting with SQL and Power BI, saving the team 10 hours per week.
- Deployed models with Docker and Flask on AWS; mentored 2 junior analysts.
Data Analyst, FinServe (2020 - 2022)
- Analyzed 2M+ transaction records with pandas and SQL to detect churn drivers.
- Created dashboards in Tableau used by 40 managers.

EDUCATION
M.Sc. in Statistics, 2020
B.Sc. in Mathematics, 2018

SKILLS
Python, SQL, machine learning, deep learning, pandas, numpy, scikit-learn, TensorFlow, statistics,
data visualization, Tableau, Power BI, Docker, Git, communication, teamwork

PROJECTS
Resume classifier using NLP and logistic regression (accuracy 91%).

In [ ]:
%%writefile samples/resume_web_developer.txt
Arjun Mehta
arjun.m@example.com | 555-201-3344 | github.com/arjun-web

EXPERIENCE
Frontend Developer, PixelWorks (2021 - 2024)
- Developed responsive websites using HTML, CSS, JavaScript and React for 15+ clients.
- Improved page load time by 35% with lazy loading and image optimisation.
- Collaborated with designers in Figma and used Git for version control.

EDUCATION
B.Tech in Computer Science, 2021

SKILLS
HTML, CSS, JavaScript, React, Bootstrap, Figma, Git, WordPress, jQuery, responsive design

In [ ]:
%%writefile samples/resume_fresher.txt
Sara Khan
sara.khan@example.com

I am a recent graduate. I am hardworking and I love learning new things. My hobbies are reading and cricket.

Education
B.Sc. Computer Science 2023-2026

I know Python and some Excel. I did a small project on student marks.

## 3. Train the best-fit-role model (optional, ~1 minute)
Only the *"best-fit role"* column needs this. The match %, skills, ATS check and ranking work without it.
It downloads the Kaggle dataset automatically; if that fails, the cell just skips training.
Duplicates are removed before splitting, so the accuracy printed is for resumes the model has never seen.

In [ ]:
import glob, shutil, subprocess, sys
CSV = "dataset/UpdatedResumeDataSet.csv"
try:
    if not os.path.exists(CSV):
        import kagglehub
        path = kagglehub.dataset_download("jillanisofttech/updated-resume-dataset")
        shutil.copy(glob.glob(os.path.join(path, "**", "UpdatedResumeDataSet.csv"), recursive=True)[0], CSV)
    print(subprocess.run([sys.executable, "train.py"], capture_output=True, text=True).stdout[-1500:])
except Exception as e:
    print("Skipping training:", type(e).__name__, e, "\n(The app still works; role suggestions will be off.)")

## 4. Quick check (no upload needed)
Runs the matcher on the built-in sample CVs so you can see the engine works before starting the app.

In [ ]:
import sys
sys.path.insert(0, os.getcwd())
from resume_ai.analyzer import analyze

jd = open("samples/jd_data_scientist.txt").read()
for f in ["resume_data_scientist", "resume_web_developer", "resume_fresher"]:
    a = analyze(open(f"samples/{f}.txt").read(), jd)
    m = a["match"]
    print(f"{f:24s} match {m['score']:5.1f}%  ({m['verdict']})   ATS {a['ats']['score']}/100   missing required: {', '.join(m['missing_required']) or 'none'}")

## 5. Launch the app and open it
This starts the Streamlit app and gives you a **public link** (Cloudflare tunnel). Open the link, then:
1. paste a job description, 2. drop your CVs (or a ZIP), 3. press **Analyze**.

Keep this cell running while you use the app. The link works only while the notebook is connected.

In [ ]:
import subprocess, sys, time, re, os, shutil, stat, threading, urllib.request

PORT = 8501
subprocess.run(["pkill", "-f", "streamlit run"], capture_output=True)
log = open("streamlit.log", "w")
app_proc = subprocess.Popen(
    [sys.executable, "-m", "streamlit", "run", "app.py", "--server.port", str(PORT), "--server.headless", "true",
     "--server.enableCORS", "false", "--server.enableXsrfProtection", "false", "--browser.gatherUsageStats", "false"],
    stdout=log, stderr=subprocess.STDOUT)

for _ in range(60):                                   # wait until the app answers
    try:
        urllib.request.urlopen(f"http://localhost:{PORT}/_stcore/health", timeout=2); break
    except Exception:
        time.sleep(1)
else:
    print(open("streamlit.log").read()[-2000:]); raise SystemExit("The app did not start - see the log above.")
print("App is running.")

url = None
try:
    if not os.path.exists("cloudflared"):
        urllib.request.urlretrieve("https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64", "cloudflared")
        os.chmod("cloudflared", os.stat("cloudflared").st_mode | stat.S_IEXEC)
    tunnel = subprocess.Popen(["./cloudflared", "tunnel", "--url", f"http://localhost:{PORT}", "--no-autoupdate"],
                              stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    t0 = time.time()
    while time.time() - t0 < 60 and url is None:
        line = tunnel.stdout.readline()
        m = re.search(r"https://[-a-z0-9]+\.trycloudflare\.com", line or "")
        if m: url = m.group(0)
    threading.Thread(target=lambda: [None for _ in tunnel.stdout], daemon=True).start()   # keep the pipe drained
except Exception as e:
    print("Tunnel problem:", e)

if url:
    print("\n👉 Open this link and upload your CVs there:\n\n   ", url, "\n")
else:
    print("Could not create a public link. Trying Colab's built-in proxy instead ...")
    from google.colab import output
    output.serve_kernel_port_as_window(PORT)

In [ ]:
# Run this cell to stop the app when you are done
app_proc.terminate()
try: tunnel.terminate()
except Exception: pass

## 6. Get your results
Everything you generate is also saved on this Colab machine in `zarshan-project/outputs/<timestamp>/` (ranking.csv, reports.zip, results.json).
Use the download buttons in the app, or the Files panel on the left (right-click > Download).

**Tips**
- Many CVs? Put them in **one ZIP** and drop that single file in the app.
- Add your own skills to `resume_ai/skills.py` (`EXTRA_SKILLS`, `ALIASES`) and rerun the Launch cell.
- To run it on your own computer: download the `zarshan-project` folder, `pip install -r requirements.txt`, then `streamlit run app.py`.